# 04: 多方法嵌入比较——PI 工作界面

运行 PCA（基线）、Harmony、scVI（从头训练），以及可选的
scANVI（存在标注参考时）和 cellxgene_census 预训练 scVI。

所有嵌入方法**等权并列**——无优先级预设。选择依据：
1. **可视化检查（首要）**——每个嵌入算完**即时出 UMAP 图**，按样本、批次、
   细胞类型三着色。PI 跑一个看一个，目视判断批次整合与生物学信号保留的平衡。
2. **整合指标（佐证）**——全部嵌入跑完后，显式 for 循环对每个嵌入做内联 silhouette_score 计算生成对比表。无回调、无 sweep() 抽象，计算过程可见可调。

**本 notebook 产出**：
- `obsm["X_pca"]`——PCA（基线，仅 HVG）
- `obsm["X_pca_harmony"]`——Harmony 批次校正 PCA
- `obsm["X_scVI"]`——scVI 潜变量（从头训练）
- `obsm["X_scANVI"]`——scANVI 潜变量（仅当存在标注参考时）
- 每个嵌入的 UMAP 图：计算后立即出图（三着色），跑一个看一个
- `results/figures/sweep_04/` 中显式遍历产出对比表（含整合指标）
- `adata.uns` 运行元数据（`harmony_v1`、`scvi_v1` 等）
- 04 `NEEDS_REVIEW` draft checkpoint；人工选择前不供 05 使用

**增加新嵌入方法**（扩展模式）：写 `adata.obsm["X_{method}"]`，
并将 `"X_{method}"` 加入整合指标 cell 的 `use_reps` 列表——一个 cell，零框架改动。

## 如何回跑（迭代调参机制）

### 管线位置
- **上游**：03（标准化 + HVG），读 `03_normalized_v*.h5ad`
- **下游**：05（多分辨率 Leiden 聚类），产出 `04_embedded_v*.h5ad`

### 为什么要迭代回跑？
嵌入质量直接影响下游分群和注释的准确性。如果在 05（Leiden 分群不合理）、
06（注释时发现嵌入没有分离应区分的细胞类型）发现问题，可能需要：
- 换用不同的嵌入方法（增减 `EMBEDDING_METHODS` 列表）
- 调整 PCA 维度数（`N_PCS`）
- 增加 scVI 训练轮数（`SCVI_MAX_EPOCHS`——默认 20 是快速验证用，生产建议 200-400）
- 换用 03 的另一个版本（不同 HVG 数量）

PI 的原始要求（来自项目构思）：
> "注释这一步，也可能在注释的过程中发现前面高可变基因的选择、embedding 的构建，
> 还有分群的参数等等需要调整，应可以随时调回去重跑一些流程，需要建立这种循环不断迭代的机制。"

### 如何回跑（三步操作）
1. **改 `UPSTREAM_RUN_ID`**——选择要复用的 Stage 03 promoted run
   （例如 `03-normalized-v2-run001`）
2. **改 `RUN_ID`**——每次调参使用新的、不可覆盖的运行标识
   （例如 `04-embedded-v2-run001`）
3. **调整参数**（在下方 `# === PARAMS ===` 区域改嵌入方法或训练参数）
   → 重跑本 notebook（Cell → Run All）

### 版本约定
- **`draft`**：本 notebook 计算出的候选嵌入，固定为 `NEEDS_REVIEW`。
- **`promoted`**：研究者审查并明确选择嵌入后，才可提升为下游权威输入。
  本 PR 不替研究者选择，也不创建 promoted 输出；选择与提升在后续 PR4 接入。
- **下游取数**：Stage 05 只能读取 promoted Stage 04；当前 draft 因而不可消费。

### 追溯链（自动写入 h5ad 的 `adata.uns`）
本 notebook 在写出前自动记录以下字段，供后续审计查询：
- `stage` = `"04_embedded"`（本 stage 标识）
- `status` = `"NEEDS_REVIEW"`（禁止手工改为 `"SUCCESS"`）
- `upstream` = 本次读入的上游文件路径列表
- `version` = 与 `OUTPUT_VERSION` 一致的版本号（`"v1"` / `"v2"` / ...）

如需查询"04 有哪些版本？哪些依赖 03_v1？"，
可直接检查 `results/runs/<RUN_ID>/draft/manifest.json`。

In [ ]:
# === PARAMS ===

UPSTREAM_RUN_ROOT = "results/runs"
UPSTREAM_RUN_ID = "03-normalized-v1-run001"
RUN_ROOT = "results/runs"
RUN_ID = "04-embedded-v1-run001"
OUTPUT_FILENAME = "04_embedded_v1.h5ad"

# --- PCA ---
N_PCS = 50                          # 多算，用 elbow plot 决定实际使用数
N_PCS_USE = 30                      # 实际送入邻居图/Harmony 的 PC 数（经验值 20-40，胃全层组织建议 30）
                                    # N_PCS=50 只是"多算"用于 elbow 诊断，N_PCS_USE 才决定下游用多少

# --- 邻居图（支持 Scalar-or-Sweep）---
# 经验法则：k ≈ sqrt(n_cells)/2，最少 10 最多 100
#   - 稀有细胞类型关注 → 小 k（10-15）
#   - 大群概览 → 大 k（30-50）
#   - 最终标准：已知 marker（如 CD3）是否在 UMAP 上连续
N_NEIGHBORS = 15            # 单值直接跑 | 列表如 [10, 15, 20, 30] → 对比 UMAP 形态
METRIC = "cosine"                   # "euclidean" | "cosine"（cosine 对高维更稳健）

# --- UMAP ---
UMAP_MIN_DIST = 0.3                 # 0.1=紧密 0.5=分散（视觉参数，不影响分析）
UMAP_SPREAD   = 1.0

# --- 嵌入方法（UX-1：研究者逐方法勾选）---
# 不设固定 profile、不强制 PCA。每方法一个布尔开关，勾选 True 即运行。
# 不满足前置条件的方法将在预检 cell 中禁用并显示原因。
# 所有被执行的方法等权并列——无优先级预设。

PCA_ENABLED = True
#   输入要求: normalized log1p X + HVG（03 产出）
#   是否需 GPU: 否（CPU 即可）
#   预计耗时: <1 分钟（~10k 细胞）
#   适用场景: 所有分析——作为无批次校正的基线对照
#   主要风险: 批次效应不做任何校正，样本按来源明显分离属正常
HARMONY_ENABLED = True
#   输入要求: X_pca 已计算 + batch_key 列存在
#   是否需 GPU: 否（CPU 即可）
#   预计耗时: <1 分钟（~10k 细胞，theta x iter 默认）
#   适用场景: 快速迭代调参、初步判断批次效应强度
#   主要风险: 过校正（theta 过高时抹平生物学差异）；非线性批次校正有限
SCVI_ENABLED = True
#   输入要求: layers["counts"] 原始整数计数（负二项模型要求）
#   是否需 GPU: 强烈建议（CPU 训练 ~10 倍慢）；无 GPU 降级 CPU 也可跑
#   预计耗时: 5-20 分钟（~10k 细胞，200 epochs，GPU）；CPU 需 50-200 分钟
#   适用场景: 复杂批次效应、非线性校正需求、深层生物学差异保留
#   主要风险: 训练不收敛（检查 ELBO 曲线）；小批次过拟合（规模差异 >10x）
SCANVI_ENABLED = True
#   输入要求: layers["counts"] + adata.obs 中有效标签列（需设 SCANVI_LABEL_KEY）
#   是否需 GPU: 同 scVI
#   预计耗时: scVI 训练 + 额外 2-5 分钟（scANVI 微调）
#   适用场景: 存在可靠细胞类型标注参考——潜空间同时感知批次与细胞类型
#   主要风险: 标签质量差或覆盖不全时不如纯 scVI；标签列 NaN 过多不可用
CENSUS_ENABLED = False
#   输入要求: cellxgene_census 包已安装 + Census 存在该组织预训练模型
#   是否需 GPU: 否（仅推理，不训练）
#   预计耗时: 取决于 Census 下载速度（首次需下载模型，约数分钟）
#   适用场景: 需要外部大规模参考嵌入空间时
#   主要风险: 预训练模型与实验协议分布偏移；Census API 不稳定
SCCRAFT_ENABLED = False
#   输入要求: layers["counts"] + scCRAFT 已安装（非 PyPI，需 GitHub 源码装）
#   是否需 GPU: 内部硬编码 CPU（本参数无效）——跨平台均可跑但慢
#   预计耗时: 较长（VAE + 判别器对抗训练，~10k 细胞需 10-30 分钟）
#   适用场景: anchor-free 整合、样本量较小时、需判别器增强批次混合
#   主要风险: 对抗训练不稳定；超参（d_coef/kl_coef）影响大；非标准安装

# --- 批次校正 key ---
BATCH_KEY = "source_dataset"
# 含义：定义"批次"的 adata.obs 列名——Harmony/scVI 会校正不同批次细胞间的技术差异，
#       让不同批次的相同细胞类型在嵌入空间中混合在一起。
# 默认依据：来源数据集（source_dataset）是最粗粒度的技术差异来源——
#           不同课题组/实验室/测序平台产生的系统性偏差。
# 调粗（"source_dataset"）影响：只校正来源级差异，同一来源内不同样本间的差异不校正。
#                               适合来源间差异明显、来源内样本同质的场景。
# 调细（"sample_id"）影响：校正每个样本的独立技术差异（如不同解离批次、测序run），
#                         但样本量少时 scVI 可能为每个样本学习独立的批次参数导致过拟合。
# 何时改：① 不同来源的数据在 UMAP 上明显分离 → 用 "source_dataset"
#         ② 同源内不同样本间仍有明显分离 → 改为 "sample_id"
#         ③ 二阶段校正（先来源后样本）→ 需跑两次本 stage 且分析复杂，超出默认范围

# --- Harmony（支持 Scalar-or-Sweep）---
HARMONY_THETA = 2.0
# 含义：Harmony 批次校正的惩罚强度系数——控制不同批次的细胞在嵌入空间中混合的程度。
#       theta=0 等同于不校正，theta 越大批次混合越强。
# 默认依据：Korsunsky et al. 2019 Nature Methods 论文推荐的中等强度值（1.5-2.5）。
# 调大（如 3-4）影响：批次混合更彻底，但可能过度校正——抹平真实的细胞类型差异，
#                     让本该分离的不同细胞类型错误地聚在一起。
# 调小（如 1-1.5）影响：保留更多生物学差异，但批次效应残留风险增大——
#                       来自不同医院/实验批次的相同细胞类型可能仍分离。
# 何时改：① 数据集来自不同测序平台或医院（强技术批次）→ 增大到 3-4
#         ② 同一实验室轻微批次差异 → 减小到 1-1.5
#         ③ 通过过校正检测 cell 发现 type_mixing > 0.3 → 考虑降低
# 单值直接跑 | 列表如 [1, 2, 3] → Sweep 对比校正强度
HARMONY_MAX_ITER = 20

# --- scVI ---
SCVI_N_LATENT    = 30
# 含义：scVI 潜空间维度——模型将每个细胞的基因表达（数千维）压缩为多少维的低维向量。
#       类似 PCA 的主成分数，但通过神经网络学习非线性压缩。
# 默认依据：与 N_PCS_USE 对齐（均为 30），便于与 Harmony/PCA 做等维对比；
#           scVI 原论文建议 10，但多细胞类型复杂组织（如肿瘤、免疫微环境）建议 20-30。
# 调大（如 40-50）影响：能捕捉更多细胞状态的细微差异，但 UMAP 可能出现更多细碎分支，
#                       训练时间增加，过拟合风险增大。
# 调小（如 10-15）影响：更简洁的表示，稀有细胞类型可能合并到主要类型中，分辨率降低。
# 何时改：① UMAP 过于细碎（出现大量无生物学意义的小簇）→ 减小到 20
#         ② 已知细胞类型在 UMAP 上分离不清晰 → 适当增大到 40
SCVI_N_LAYERS    = 2
SCVI_N_HIDDEN    = 128
SCVI_MAX_EPOCHS  = 200              # 生产 200-400；快测 50
SCVI_EARLY_STOPPING = True
SCVI_GENE_LIKELIHOOD = "zinb"

# --- 计算设备（device 自适应：根据 CUDA/MPS/CPU 可用性自动选择）---
# "auto" 自动：CUDA GPU > (Mac)scVI/scANVI 用 CPU > CPU；可显式 "cuda"|"mps"|"cpu"
# 注意：scCRAFT 内部硬编码 CPU，本参数对 scCRAFT 无效
DEVICE = "auto"

# --- scANVI ---
SCANVI_N_EPOCHS = 100
SCANVI_LABEL_KEY = None             # None = 不运行 scANVI（需显式设置标签列名才启用）

# --- scCRAFT（需单独安装: git clone https://github.com/ch2343/scCRAFT && pip install .）---
SCCRAFT_RESOLUTION    = 0.5         # 低分辨率聚类系数
SCCRAFT_CLUSTER_METHOD = "Leiden"   # "Louvain"（快）或 "Leiden"（慢但更准）
SCCRAFT_EPOCHS        = 150         # batch 数 > 80 时降到 50
SCCRAFT_WARMUP_EPOCH  = 50          # 约为 epochs 的 1/3
SCCRAFT_D_COEF        = 0.2         # 判别器损失系数（越高批次混合越强）
SCCRAFT_KL_COEF       = 0.005       # KL 散度比例（batch effect 小时降到 0.0005 更好保留细胞身份）
SCCRAFT_N_TOP_GENES   = 2000        # scCRAFT 内部 HVG 选择数量

# --- 研究者选择（UX-1 决策 cell，不自动赋值）---
SELECTED_EMBEDDING = None            # 研究者显式选择后填入（如 "X_scVI"）
SELECTION_RATIONALE = ""              # 选择理由（可选，一句话记录）

OUTPUT_VERSION = 1
# === Visualization ===
MARKER_GENES_UMAP = [
    "EPCAM",    # 上皮细胞标志基因
    "PTPRC",    # 免疫细胞标志基因（CD45）
    "COL1A1",   # 成纤维细胞标志基因
    "PECAM1",   # 内皮细胞标志基因（CD31）
]
# 含义：用于评估嵌入空间是否保留生物学信号的已知 marker 基因列表。
#       会在 UMAP 上叠加展示这些基因的表达，检查不同细胞类型是否正确分离。
# 默认依据：胃癌组织的主要细胞类型 marker（上皮/免疫/间质/内皮）。
# 何时改：切换到其他组织类型时，需替换为对应的 marker 基因。
#         例如：脑组织 → ["RBFOX3"(神经元), "GFAP"(星形胶质), "MBP"(少突)]
#               肝组织 → ["ALB"(肝细胞), "KRT19"(胆管), "CLEC4F"(Kupffer)]

RANDOM_SEED    = 42

# --- 跨 cell 状态变量预初始化（防止 cell 跳执行时下游 NameError）---
_counts_key = None
_counts_source = None
_converged = True
# 决策4 scVI 严格校验结果（scVI/scANVI/scCRAFT 训练前由校验 cell 设置）
_counts_valid = False
_counts_failure_reasons = []
_method_status = {}  # per-method 状态（success/skipped_by_user/unavailable/failed）

In [ ]:
# 确保框架 src/ 在 sys.path 上，CWD 为项目根目录。
# 自动检测两种运行场景：从 notebooks/（Jupyter）还是项目根目录（nbconvert）启动。
import sys, os, json
from pathlib import Path
_root = os.getcwd()
# 向上逐级查找项目根（含 src/scrna_integration 的目录），兼容任意嵌套深度
while _root != os.path.dirname(_root):
    if os.path.isdir(os.path.join(_root, "src", "scrna_integration")):
        break
    _root = os.path.dirname(_root)
if os.path.join(_root, "src") not in sys.path:
    sys.path.insert(0, os.path.join(_root, "src"))
os.chdir(_root)
os.makedirs("results/figures", exist_ok=True)
os.makedirs("results/figures/sweep_04", exist_ok=True)
print(f"PROJECT_ROOT: {_root}")

# A800 64核 OpenBLAS默认全开致线程爆炸（200+线程冻结），限制为4
os.environ.setdefault("OPENBLAS_NUM_THREADS", "4")
os.environ.setdefault("OMP_NUM_THREADS", "4")
os.environ.setdefault("MKL_NUM_THREADS", "4")
os.environ.setdefault("NUMBA_NUM_THREADS", "4")

# 导入（scanpy 原生 API + 框架函数仅在真正需要时使用）。
import scanpy as sc
import scipy.sparse as sp
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime
import re
import warnings
from scrna_integration.run_contract import (
    MethodStatus, StageStatus, aggregate_method_status, atomic_write_json,
    collect_runtime_provenance, determine_stage_status, prepare_run,
    resume_run, sha256_file, snapshot_effective_parameters,
    validate_checkpoint, validate_expression_contract,
)

# 整合指标计算已内联至本 notebook cell（不再从 scorers import，符合 src/notebook 边界铁律）
from sklearn.metrics import silhouette_score

# 抑制 scvi-tools PyTorch Lightning 弃用警告
warnings.filterwarnings("ignore", message=".*Lightning.*")
warnings.filterwarnings("ignore", message=".*The number of training batches.*")

sc.settings.verbosity = 2
sc.settings.set_figure_params(dpi=100, facecolor="white", frameon=False)
sc.settings.figdir = "results/figures"

import importlib.metadata
print(f"Scanpy {importlib.metadata.version('scanpy')}  |  anndata {importlib.metadata.version('anndata')}")

# === 只读取 promoted Stage 03 上游 ===
upstream_run = resume_run(UPSTREAM_RUN_ROOT, UPSTREAM_RUN_ID, promoted=True)
upstream_manifest_path = upstream_run.promoted_dir / "manifest.json"
upstream_manifest = json.loads(upstream_manifest_path.read_text(encoding="utf-8"))
if upstream_manifest.get("run_id") != UPSTREAM_RUN_ID:
    raise ValueError("upstream manifest run_id 与配置不匹配")
if upstream_manifest.get("stage") != "03_normalized":
    raise ValueError("upstream manifest stage 必须是 03_normalized")
upstream_stage_status = upstream_manifest.get("stage_status")
if upstream_stage_status == "SUCCESS_WITH_WARNINGS":
    warning_acceptance = upstream_manifest.get("warning_acceptance")
    if not isinstance(warning_acceptance, dict) or not all(
        isinstance(warning_acceptance.get(field), str) and warning_acceptance[field].strip()
        for field in ("accepted_by", "accepted_at")
    ):
        raise ValueError("upstream SUCCESS_WITH_WARNINGS 必须有明确的 warning_acceptance")
elif upstream_stage_status != "SUCCESS":
    raise ValueError(f"upstream stage_status 不可用于下游: {upstream_stage_status!r}")
UPSTREAM_CHECKPOINT = validate_checkpoint(upstream_manifest_path)
upstream_input = {
    "run_id": UPSTREAM_RUN_ID, "stage": "03_normalized",
    "manifest_path": str(upstream_manifest_path),
    "manifest_sha256": sha256_file(upstream_manifest_path),
    "checkpoint_path": str(UPSTREAM_CHECKPOINT),
    "checkpoint_sha256": sha256_file(UPSTREAM_CHECKPOINT),
}
print("加载已验证上游:", UPSTREAM_CHECKPOINT)
adata = sc.read_h5ad(UPSTREAM_CHECKPOINT)
# === promoted Stage 03 上游读取完成 ===
print(f"已加载: {adata.n_obs:,} 细胞 x {adata.n_vars:,} 基因")
print(f"X dtype: {adata.X.dtype}  |  sparse: {sp.issparse(adata.X)}")
print(f"layers: {list(adata.layers.keys())}")
print(f"obsm keys: {list(adata.obsm.keys())}")

# 确定 UMAP 着色列（全程复用）。
# sample_id——样本是否混合均匀（批次整合质量）
# BATCH_KEY——声明的批次变量是否被校正
# 细胞类型列——已知细胞类型是否被分离（生物学信号保留）
colour_columns = ["sample_id"]
if BATCH_KEY in adata.obs.columns and BATCH_KEY != "sample_id":
    colour_columns.append(BATCH_KEY)
for ct_cand in ["Celltypes_global", "cell_type", "cell_type_original"]:
    if ct_cand in adata.obs.columns:
        colour_columns.append(ct_cand)
        break
print(f"UMAP 着色列: {colour_columns}")

# 环境自检（每次运行自动检测平台 + conda 环境 + 关键包）
try:
    from scrna_integration.platform import env_check
    env_check()
except Exception as _e:
    print(f"环境自检跳过（env_check 不可用: {_e}）")

# 计算设备概览（device 自适应：根据 CUDA/MPS/CPU 可用性自动选择）
from scrna_integration.platform import detect_device
_dev_overview = detect_device(prefer=DEVICE)
print(f"计算设备: {_dev_overview['device_str']}  ({_dev_overview['reason']})")

In [ ]:
# === Counts 契约验证（决策 1/2：layers["counts"] 是唯一权威 counts）===
# scVI/scANVI 依赖原始整数 counts 做负二项建模。进入 04 后不再从 .raw
# 临时补救——.raw→counts 提取必须在 01 完成。本 cell 验证
# expression_contract，不合格时标记 counts 依赖方法为 unavailable，
# 不静默补建 counts。
from scrna_integration.run_contract import validate_expression_contract, MethodStatus

try:
    # 验证 expression_contract：X 应为 normalized_log1p，由 stage 03 产出
    _contract = validate_expression_contract(adata, expected_scale="normalized_log1p", stage="03")

    # 验证 layers["counts"] 存在且与契约一致
    _counts_layer = _contract["counts_layer"]
    if _counts_layer not in adata.layers:
        raise KeyError(
            f"layers[{_counts_layer!r}] 声明于 expression_contract "
            f"但不存在于 adata.layers: {list(adata.layers.keys())}"
        )

    _counts_key = _counts_layer
    _counts_source = _contract["counts_source"]

    print("✓ expression_contract 验证通过")
    print(f"  X 尺度: {_contract['x_scale']}")
    print(f"  counts 来源: {_counts_source}")
    print(f"  counts 层: layers[{_counts_key!r}]")
    print(f"  counts 整数验证: {_contract['counts_integer_check']}")
    print(f"  来源 stage: {_contract['stage']}")
    print(f"  SoupX 层: {_contract['soupx_layer']}")

except (KeyError, ValueError) as _e:
    print(f"❌ expression_contract 验证失败: {_e}")
    print("   scVI/scANVI/scCRAFT 将标记为 unavailable，不再静默补建 counts。")
    _counts_key = None
    _counts_source = None
    _counts_valid = False
    _counts_failure_reasons = ["expression_contract 验证失败"]

In [ ]:
# === 批次规模诊断 ===
# 极端不平衡的批次规模（>10x）会导致 scVI 对小批次过拟合、
# Harmony 校正不足等问题。提前报告让 PI 知情。
if BATCH_KEY in adata.obs.columns:
    _batch_counts = adata.obs[BATCH_KEY].value_counts()
    print(f"\n===== 批次规模（{BATCH_KEY}）=====")
    for b, n in _batch_counts.items():
        print(f"  {b}: {n:,}")
    _ratio = _batch_counts.max() / _batch_counts.min()
    print(f"  最大/最小比: {_ratio:.1f}x")
    if _ratio > 10:
        print("  ⚠️ 批次规模差异 > 10 倍")
        print("  → scVI 可能对小批次过拟合，建议减小 batch_size 或对大批次 downsample")
    elif _ratio > 5:
        print("  △ 批次规模差异 5-10 倍，注意观察 scVI UMAP 中小批次是否过度分离")
    else:
        print("  ✓ 批次规模均衡")
else:
    print(f"\n===== 批次规模诊断（跳过——BATCH_KEY='{BATCH_KEY}' 不在 obs 列中）=====")

In [ ]:
# === scVI/scANVI 输入严格校验（决策 4）===
# scVI/scANVI 默认只读 layers["counts"]（负二项模型针对未归一化计数）。
# 训练前执行六项严格校验。任意一项失败即标记所有 counts 依赖方法为 unavailable，
# 不静默改用 .raw、不静默降级 PCA/Harmony。

# 仅当至少一种 counts 依赖方法（scvi/scanvi/sccraft）被勾选时才执行此校验；
# 若均未勾选，跳过以节省时间。
if SCVI_ENABLED or SCANVI_ENABLED or SCCRAFT_ENABLED:
    print("\n===== scVI/scANVI 输入严格校验（决策 4）=====\n")
    _counts_valid = False
    _counts_failure_reasons = []

    # 预初始化六项校验结果为 False。
    # 为什么必须在此预定义：当 counts 层不可用（下方 if 分支）时，六项实检
    # 被整体跳过，但块末尾 adata.uns["scvi_validation"] 仍要读取这些变量做
    # 持久化。不预定义则边界路径会抛 NameError，反而绕过决策 4 的阻断语义。
    _check1 = _vnames_correct = _check3 = _check4 = _check5 = _check6 = False

    if _counts_key is None or _counts_key not in adata.layers:
        _counts_failure_reasons.append(
            f"counts 层不可用: _counts_key={_counts_key!r}, layers={list(adata.layers.keys())}")
        _counts_valid = False
    else:
        # 加载 counts 矩阵（稀疏保持，不 densify——避免内存溢出）
        _c = adata.layers[_counts_key]
        print(f"counts 层: layers[{_counts_key!r}]  shape={_c.shape}  dtype={_c.dtype}")

        # --- 校验 1：shape 与 obs/var 一致 ---
        _check1 = (_c.shape[0] == adata.n_obs and _c.shape[1] == adata.n_vars)
        if not _check1:
            _counts_failure_reasons.append(
                f"shape 不一致: counts={_c.shape} vs (obs={adata.n_obs}, var={adata.n_vars})")
        print(f"  校验 1 shape 一致: {'PASS' if _check1 else 'FAIL'}")

        # --- 校验 2：counts 基因轴与 var_names 严格对齐（决策 4 核心）---
        # scVI 从 layers[counts] 逐基因读取原始计数做负二项建模，其基因轴
        # 必须与 adata.var_names 严格对齐——否则会把 A 基因的计数当作 B 基因
        # 训练，静默产出错位的错误嵌入（无报错、结果全错，最危险）。
        # 两项真实校验：
        #   2a 基因轴宽度：counts 的列数必须等于 var_names 的长度。若上游发生
        #      subset/重排、或 n_vars 与 var_names 失同步，此项会捕获漂移。
        #   2b 基因名唯一：var_names 存在重复时，基于基因名的索引产生歧义，
        #      scVI 的 setup_anndata 可能取错列，同样导致静默错位。
        _var_names = list(adata.var_names)
        _axis_match = (_c.shape[1] == len(_var_names))
        _no_dup_genes = (len(_var_names) == len(set(_var_names)))
        _vnames_correct = _axis_match and _no_dup_genes
        if not _axis_match:
            _counts_failure_reasons.append(
                f"counts 基因轴({_c.shape[1]}) 与 var_names({len(_var_names)}) 长度不一致——基因错位")
        if not _no_dup_genes:
            _n_dup = len(_var_names) - len(set(_var_names))
            _counts_failure_reasons.append(
                f"var_names 含 {_n_dup} 个重复基因名——基因轴索引歧义")
        print(f"  校验 2 基因顺序: {'PASS' if _vnames_correct else 'FAIL'}")

        # --- 校验 3：全部值有限非负 ---
        # 对于稀疏矩阵，检查 data 数组（非零元素）——零元素必然是有限的
        if sp.issparse(_c):
            _data = _c.data
            _check3 = bool(np.isfinite(_data).all()) and bool((_data >= 0).all())
        else:
            _check3 = bool(np.isfinite(_c).all()) and bool((_c >= 0).all())
        if not _check3:
            _counts_failure_reasons.append("counts 含 NaN/Inf 或负值")
        print(f"  校验 3 有限非负: {'PASS' if _check3 else 'FAIL'}")

        # --- 校验 4：近整数检查（分块，避免一次性 densify 大矩阵）---
        # scVI 负二项模型假设整数 counts；浮点截断误差可接受（< 1e-4），
        # 但明显非整数（如 normalized 值）必须阻断。
        _check4 = True
        _block_size = 1000  # 每块 1000 个基因，blockwise 检查
        if sp.issparse(_c):
            _n_genes = _c.shape[1]
            _n_blocks = (_n_genes + _block_size - 1) // _block_size
            _sample_checked = False
            for _bi in range(_n_blocks):
                _start = _bi * _block_size
                _end = min(_start + _block_size, _n_genes)
                _block = _c[:, _start:_end]
                if sp.issparse(_block):
                    _block = _block.toarray()
                # 每块取前 500 个细胞（避免全量 densify）
                _n_check = min(500, _block.shape[0])
                _sample = _block[:_n_check, :]
                if _sample.size > 0:
                    _sample_checked = True
                    _rounded = np.round(_sample)
                    if not np.allclose(_sample, _rounded, rtol=0, atol=1e-4):
                        _max_diff = np.abs(_sample - _rounded).max()
                        _check4 = False
                        _counts_failure_reasons.append(
                            f"counts 非整数（max|diff|={_max_diff:.2e} > 1e-4）")
                        break
            if not _sample_checked and _c.nnz > 0:
                _sample = _c.data[:min(500, _c.nnz)]
                _rounded = np.round(_sample)
                if not np.allclose(_sample, _rounded, rtol=0, atol=1e-4):
                    _check4 = False
                    _counts_failure_reasons.append("counts 非整数（sparse data 抽检）")
        print(f"  校验 4 近整数（blockwise）: {'PASS' if _check4 else 'FAIL'}")

        # --- 校验 5：每批次有有效文库大小 ---
        # 计算每细胞总 counts（行和），按批次分组检查。
        # 某个批次的所有细胞文库大小全为零 → 该批次数据异常。
        _check5 = True
        if BATCH_KEY in adata.obs.columns:
            _batch_lib = {}
            for _b in adata.obs[BATCH_KEY].unique():
                _mask = (adata.obs[BATCH_KEY] == _b).values
                if sp.issparse(_c):
                    _lib = np.asarray(_c[_mask].sum(axis=1)).flatten()
                else:
                    _lib = _c[_mask].sum(axis=1)
                _nonzero = (_lib > 0).sum()
                _batch_lib[_b] = {"n_cells": int(_mask.sum()), "nonzero_lib": int(_nonzero)}
                if _nonzero == 0:
                    _check5 = False
                    _counts_failure_reasons.append(
                        f"批次 {_b!r}: {int(_mask.sum())} 个细胞全部零文库")
            print(f"  校验 5 每批次有效文库: {'PASS' if _check5 else 'FAIL'}")
            for _b, _info in _batch_lib.items():
                print(f"    {_b}: {_info['nonzero_lib']}/{_info['n_cells']} 有效文库")
        else:
            # BATCH_KEY 缺失时本项无法执行——不能默认 PASS 掩盖问题，
            # 但也不阻断（批次校正是可选前提）。保持 True 表示"未发现零文库批次"。
            print(f"  校验 5 每批次有效文库: 跳过（BATCH_KEY={BATCH_KEY!r} 不在 obs 中）")

        # --- 校验 6：契约元数据确认来源与处理历史 ---
        # expression_contract 记录 counts 来源（X/.raw.X/layers[counts]）、
        # 来源 stage、处理历史——确认 counts 确实是原始未归一化数据。
        _check6 = True
        if _counts_source is not None:
            print(f"  校验 6 契约元数据: counts_source={_counts_source}")
            if _counts_source == "X":
                # X 是 normalized log1p（03 产出），不作为 scVI 输入——异常
                _check6 = False
                _counts_failure_reasons.append(
                    f"counts_source={_counts_source!r}，X 是 normalized 不是原始 counts")
            print(f"  校验 6 契约元数据: {'PASS' if _check6 else 'FAIL'}")
        else:
            print("  校验 6 契约元数据: PASS（counts_source 来自 expression_contract）")

        # 汇总判断（六项全 True 才通过；校验 2 现由真实的基因轴对齐检查驱动）
        _all_checks = [_check1, _vnames_correct, _check3, _check4, _check5, _check6]
        _counts_valid = all(_all_checks)

    if _counts_valid:
        print(f"\n✓ 六项校验全部通过，counts 层可用于 scVI/scANVI/scCRAFT")
        _counts_failure_reasons = []
    else:
        print(f"\n✗ counts 校验失败 ({len(_counts_failure_reasons)} 项):")
        for _r in _counts_failure_reasons:
            print(f"  - {_r}")
        print("  → scVI/scANVI/scCRAFT 将标记为 unavailable（决策 4：阻断裂项，不静默降级）")
        # per-method status 在各自执行 cell 中设置；此处只设全局标记

    # 持久化校验结果到 adata.uns——下游/审计可查
    adata.uns["scvi_validation"] = {
        "counts_valid": _counts_valid,
        "failure_reasons": list(_counts_failure_reasons),
        "checks": {
            "shape": _check1,
            "gene_order": _vnames_correct,
            "finite_nonnegative": _check3,
            "near_integer": _check4,
            "library_size": _check5,
            "contract_metadata": _check6,
        },
    }
else:
    print("\nscVI/scANVI 输入校验跳过（counts 依赖方法均未勾选）")
    _counts_valid = False  # 未启用时默认 False（表示未用，不算 failure）
    _counts_failure_reasons = []

print(f"_counts_valid = {_counts_valid}")

In [ ]:
# === 方法可运行性预检（UX-1）===
# 展示每种方法的前置条件满足状态。不满足条件的方法显示禁用原因。
# 此 cell 仅做信息展示——不改变 PARAMS 中的勾选值，不阻止执行。
# 勾选但不满足条件的方法将在各自执行 cell 开头标记 unavailable（决策 9）。

print("\n===== 方法可运行性预检 =====\n")

# 确定每种方法的前置条件状态
_pca_ok = True  # PCA 无额外前置——HVG 在 03 已确保
_harmony_ok = ("X_pca" in adata.obsm) or PCA_ENABLED  # 需 PCA 先跑
_harmony_batch_ok = (BATCH_KEY in adata.obs.columns)
_scvi_ok = _counts_valid  # 依赖决策4 counts 校验
_scanvi_ok = _counts_valid and (SCANVI_LABEL_KEY is not None)
_scanvi_label_ok = (SCANVI_LABEL_KEY is not None and
                    SCANVI_LABEL_KEY in adata.obs.columns) if SCANVI_LABEL_KEY else False
_census_ok = False  # cellxgene_census 需单独安装且 Census 有对应组织模型
_sccraft_ok = _counts_valid  # 依赖决策4 counts 校验

# 检查 scCRAFT 是否已安装
try:
    import scCRAFT  # noqa: F401
    _sccraft_installed = True
except ImportError:
    _sccraft_installed = False

print(f"{'方法':<12} {'勾选':<6} {'前置条件':<20} {'状态':<20}")
print("-" * 60)

for _method, _enabled, _ok, _reason in [
    ("pca",       PCA_ENABLED,       _pca_ok,          "" if _pca_ok else "(无)"),
    ("harmony",   HARMONY_ENABLED,   _harmony_ok and _harmony_batch_ok,
     "" if (_harmony_ok and _harmony_batch_ok) else
     ("需 PCA 先跑" if not _harmony_ok else f"BATCH_KEY={BATCH_KEY!r} 不在 obs")),
    ("scvi",      SCVI_ENABLED,      _scvi_ok,
     "" if _scvi_ok else f"counts 校验未通过: {'; '.join(_counts_failure_reasons)}"),
    ("scanvi",    SCANVI_ENABLED,    _scanvi_ok and _scanvi_label_ok,
     "" if (_scanvi_ok and _scanvi_label_ok) else
     (f"SCANVI_LABEL_KEY 未设或无效" if not _scanvi_label_ok else
      f"counts 校验未通过: {'; '.join(_counts_failure_reasons)}")),
    ("census",    CENSUS_ENABLED,    False,  # 始终 disabled，需 Census 模型
     "需 cellxgene_census 包 + Census 托管该组织模型"),
    ("sccraft",   SCCRAFT_ENABLED,   _sccraft_ok and _sccraft_installed,
     "" if (_sccraft_ok and _sccraft_installed) else
     ("scCRAFT 未安装" if not _sccraft_installed else
      f"counts 校验未通过: {'; '.join(_counts_failure_reasons)}")),
]:
    if _enabled and not _ok:
        _status = "勾选但前置不满足"
    elif _enabled and _ok:
        _status = "将运行"
    elif not _enabled:
        _status = "未勾选"
    else:
        _status = "未知"
    print(f"{_method:<12} {'是' if _enabled else '否':<6} {('✓' if _ok else '✗'):<20} {_status:<20}")
    if _reason:
        print(f"  {'':>18} 原因: {_reason}")

print()
_any_enabled = PCA_ENABLED or HARMONY_ENABLED or SCVI_ENABLED or SCANVI_ENABLED or SCCRAFT_ENABLED
if not _any_enabled:
    print("⚠️ 所有嵌入方法均未勾选——本 stage 不会产出任何嵌入。")
    print("  请在 PARAMS cell 中至少勾选一种方法后重新运行。")
# preflight 完成标记（显式状态标记，便于调试和中断重跑）
_preflight_completed = True
print("预检完成。下方 cell 将逐一执行勾选的方法。")

## PCA（基线）

在高可变基因（HVG）上做主成分分析。这是最简单的嵌入方法——
无批次校正、无深度模型。作为所有整合方法的基线对照。

In [ ]:
# 在高可变基因（HVG）上做主成分分析。
if PCA_ENABLED:
    print(f"\n===== PCA: n_comps={N_PCS} =====")
    # 防御性检查：确保 HVG 标记存在且数量 > 0
    _n_hvg = adata.var.get("highly_variable", pd.Series(dtype=bool)).sum()
    assert _n_hvg > 0, (
        f"HVG 数量为 0——'highly_variable' 列全 False 或不存在。\n"
        f"请检查 stage 03 的 HVG 标记是否正确写入 adata.var。"
    )
    print(f"✓ PCA 将使用 {_n_hvg:,} 个高变异基因（adata.var['highly_variable'] = True）")

    sc.tl.pca(adata, n_comps=N_PCS, use_highly_variable=True,
              svd_solver="arpack", random_state=RANDOM_SEED)
    print(f"obsm['X_pca'] shape: {adata.obsm['X_pca'].shape}")
    var_explained = adata.uns['pca']['variance_ratio'].sum() * 100
    print(f"累计方差解释率 (前 {N_PCS} PCs): {var_explained:.1f}%")
    _method_status["pca"] = MethodStatus.SUCCESS
else:
    _method_status["pca"] = MethodStatus.SKIPPED_BY_USER
    print("PCA 跳过（PCA_ENABLED 为 False）")

In [ ]:
# --- PCA Elbow Plot：经验选择维度数 ---
# 方差解释率随 PC 数递减。拐点（elbow）之后的 PC 主要是噪声。
# 仅当 PCA 已计算时运行。
if "X_pca" in adata.obsm:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    # 左图：方差解释率
    # scanpy >=1.10 移除了 ax= 参数，改用 plt.sca() 设置当前轴
    plt.sca(axes[0])
    sc.pl.pca_variance_ratio(adata, n_pcs=N_PCS, log=False, show=False)
    axes[0].set_title("PCA 方差解释率")

    # 右图：累积方差
    var_ratio = adata.uns["pca"]["variance_ratio"][:N_PCS]
    cumulative = np.cumsum(var_ratio)
    axes[1].plot(range(1, N_PCS + 1), cumulative, "o-", markersize=3)
    axes[1].axhline(0.9, color="red", linestyle="--", alpha=0.5, label="90% 累积方差")
    axes[1].set_xlabel("PC")
    axes[1].set_ylabel("累积方差解释率")
    axes[1].set_title("累积方差（红线=90%）")
    axes[1].legend()

    plt.tight_layout(); plt.show()

    # 自动检测 elbow（二阶差分最大处）
    diff2 = np.diff(var_ratio, 2)
    elbow_pc = np.argmax(np.abs(diff2)) + 2  # +2 因为两次 diff 丢 2 个点
    print(f"建议 N_PCS: ~{elbow_pc}（二阶差分拐点）")
    pc90 = np.searchsorted(cumulative, 0.9) + 1
    print(f"   当前使用 N_PCS={N_PCS}，90% 累积方差在 PC {pc90}")
    # N_PCS_USE 实际应用提醒：elbow 用于诊断整个 PCA 空间，但下游只截用前 N_PCS_USE 维
    print(f"   实际使用: N_PCS_USE={N_PCS_USE}（送入邻居图/Harmony）")
    if N_PCS_USE < elbow_pc:
        print(f"   ⚠️ N_PCS_USE({N_PCS_USE}) < elbow 建议({elbow_pc})——可考虑增大")
    elif N_PCS_USE > pc90:
        print(f"   ⚠️ N_PCS_USE({N_PCS_USE}) > 90%方差({pc90})——可能引入噪音维度")
else:
    print("PCA 未计算，跳过 Elbow Plot。")

## PCA Loading 分析

检查前 3 个主成分由哪些基因驱动。如果技术基因（MT- / RPS / RPL）在 top loading 中
占据显著位置，说明 03 的 HVG 排除列表并未真正生效——这些技术基因仍通过 PCA 传递到
下游，需要回 03 调整排除参数或启用 `regress_out`。


In [ ]:
if "PCs" not in adata.varm:
    print("PCA 未运行（PCA_ENABLED 为 False），跳过 loading 分析")
else:
    # PCA loading 分析：前 3 个 PC 由哪些基因驱动？
    # 如果技术基因（MT-/RPS/RPL）主导 → 说明 03 排除列表没生效，需回 03 调整
    print("===== PCA Loading 分析（前 3 个 PC 的 top 10 genes）=====")
    loadings = pd.DataFrame(
        adata.varm["PCs"][:, :3],
        index=adata.var_names,
        columns=[f"PC{i+1}" for i in range(3)]
    )
    for pc in loadings.columns:
        top_pos = loadings[pc].nlargest(5)
        top_neg = loadings[pc].nsmallest(5)
        print(f"\\n{pc} top positive: {', '.join(top_pos.index)}")
        print(f"{pc} top negative: {', '.join(top_neg.index)}")

    # 检查技术基因是否主导 PC1-3
    tech_genes_in_top = []
    for pc in loadings.columns:
        top10 = loadings[pc].abs().nlargest(10).index
        mt_count = sum(g.startswith("MT-") for g in top10)
        ribo_count = sum(g.startswith(("RPS", "RPL")) for g in top10)
        if mt_count >= 3 or ribo_count >= 3:
            tech_genes_in_top.append(f"{pc}: MT={mt_count}, Ribo={ribo_count}")
    if tech_genes_in_top:
        print(f"\\n⚠️ 技术基因主导 PCA: {tech_genes_in_top}")
        print("  → 建议回 03 启用 EXCLUDE_MT_FROM_HVG / REGRESS_OUT")
    else:
        print("\\n✓ 前 3 个 PC 未被技术基因主导")

    # N_NEIGHBORS 经验建议
    _suggested_k = max(10, min(100, int(np.sqrt(adata.n_obs) / 2)))
    print(f"📊 经验建议 N_NEIGHBORS: {_suggested_k}（sqrt({adata.n_obs:,})/2，当前使用 k={N_NEIGHBORS if not isinstance(N_NEIGHBORS, list) else N_NEIGHBORS[-1]}）")

    # --- Stress PC 检测：解离诱导应激基因是否主导某个 PC ---
    # 组织解离过程会激活即时早期基因（IEG）和热休克蛋白，这些"应激
    # 特征"如果集中在某个 PC 上主导，会在下游 UMAP 中形成与生物学
    # 无关的簇。提前发现，可在 stage 03 做 regress_out 或在 Harmony
    # 中对该 PC 加强校正。
    _stress_genes = {"FOS", "JUN", "JUNB", "JUND", "ATF3", "EGR1", "HSPA1A", "HSPA1B",
                     "DUSP1", "ZFP36", "IER2", "NR4A1", "FOSB", "KLF2", "KLF4"}
    print(f"\n--- Stress PC 检测（解离应激基因）---")
    _stress_pcs = []
    for pc_idx in range(min(5, N_PCS)):
        _loading = adata.varm["PCs"][:, pc_idx]
        _top20_idx = np.argsort(np.abs(_loading))[-20:]
        _top20_genes = set(str(g).upper() for g in adata.var_names[_top20_idx])
        _overlap = _top20_genes & _stress_genes
        if len(_overlap) >= 3:
            _stress_pcs.append(pc_idx + 1)
            print(f"  ⚠️ PC{pc_idx+1} 含 {len(_overlap)} 个应激基因: {sorted(_overlap)}")
    if _stress_pcs:
        print(f"  → 如 UMAP 上观察到应激驱动的独立 cluster，可考虑:")
        print(f"    (a) 在 stage 03 对 S_score/G2M_score + 应激评分做 regress_out")
        print(f"    (b) 增大 Harmony theta 让其在这些 PC 上更强校正")
        print(f"    (c) 排除这些 PC（高级用法，需谨慎）")
    else:
        print(f"  ✓ 前 5 个 PC 无显著应激基因主导")

### PCA UMAP（即时查看）

PCA 是最简单的嵌入——无批次校正。作为后续方法的**基线对照**。

**怎么看**：
- 不同样本的细胞是否明显分离？→ 批次效应强，需要校正
- 已知细胞类型是否大致分开？→ 生物学信号在 PCA 空间是否可辨

对比后续 Harmony / scVI 的 UMAP，判断批次校正是否改善了混合。

In [ ]:
# PCA UMAP —— 即时查看嵌入质量
embed_key = "X_pca"
if embed_key in adata.obsm:
    print(f"\n===== PCA UMAP =====")
        # N_NEIGHBORS 支持 Scalar-or-Sweep：列表取最后一个，单值直接用
    _k_umap = N_NEIGHBORS if not isinstance(N_NEIGHBORS, list) else N_NEIGHBORS[-1]
    sc.pp.neighbors(adata, use_rep=embed_key,
                    n_pcs=min(N_PCS_USE, adata.obsm[embed_key].shape[1]),
                    n_neighbors=_k_umap, metric=METRIC,
                    random_state=RANDOM_SEED)
    sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
               random_state=RANDOM_SEED)

    # 保存 UMAP 坐标到专属 key，后续指标计算可复用。
    adata.obsm["X_umap_pca"] = adata.obsm["X_umap"].copy()

    # 三着色出图：样本、批次、细胞类型（如有）。
    for colour in colour_columns:
        if colour in adata.obs.columns:
            sc.pl.umap(adata, color=colour, title=f"PCA - {colour}",
                       frameon=False, save=f"_04_pca_{colour}.png")
            plt.close("all")

    # doublet score 着色（如果 doublet_score 可用）
    if "doublet_score" in adata.obs.columns and adata.obs["doublet_score"].notna().any():
        sc.pl.umap(adata, color="doublet_score", title=f"PCA - doublet_score",
                   frameon=False, save="_04_pca_doublet_score.png")
        plt.close("all")

    print("PCA UMAP 完成。")
else:
    print("PCA 嵌入不存在，跳过 UMAP。")

## Harmony

Harmony 通过迭代校正 PCA 嵌入来整合批次。速度快、确定性算法，
在单细胞分析管线中广泛使用。与 PCA 共用相同 `N_PCS` 参数，
batch key 来自 PARAMS 单元格。

In [ ]:
# Harmony 在 PCA 嵌入上做批次整合。
# 注意：直接使用 harmonypy（而非 sc.external.pp.harmony_integrate），
# 因为 harmonypy >= 2.0.0 把 Z_corr 方向从 (dims, cells) 改为
# (cells, dims)；scanpy wrapper 仍假设旧布局做 .T 导致 shape 不匹配。
# 故本 cell 直接调用 harmonypy 避免 scanpy wrapper 的兼容问题。
if HARMONY_ENABLED:
    if BATCH_KEY not in adata.obs.columns:
        print(f"WARNING: BATCH_KEY='{BATCH_KEY}' 不在 obs 列中，可用列: {list(adata.obs.columns)}；跳过 Harmony")
        _method_status["harmony"] = MethodStatus.UNAVAILABLE
    elif adata.obs[BATCH_KEY].nunique() < 2:
        import warnings

        _n_batches = adata.obs[BATCH_KEY].nunique()
        warnings.warn(
            f"检测到 BATCH_KEY='{BATCH_KEY}' 仅包含 {_n_batches} 个唯一值，"
            f"不存在批次间差异需要校正。Harmony 在单批次数据上运行时不会改变嵌入结果——"
            f"PCA 空间本身就没有批次效应可校正。"
            f"建议：将 HARMONY_ENABLED 设为 False；后续嵌入选择可直接使用 PCA 作为基线。",
            UserWarning,
        )
        print(f"WARNING: BATCH_KEY='{BATCH_KEY}' 仅 1 个批次，无批次差异可校正；自动跳过 Harmony")
        _method_status["harmony"] = MethodStatus.UNAVAILABLE
    elif "X_pca" not in adata.obsm:
        print("Harmony 跳过: PCA 未运行，无 X_pca 可用。")
        _method_status["harmony"] = MethodStatus.UNAVAILABLE
    else:
        print(f"\n===== Harmony: batch_key='{BATCH_KEY}' =====")
        import harmonypy

        # 只喂前 N_PCS_USE 个 PC 给 Harmony——后端 PC 含高噪声，会稀释校正信号
        ho = harmonypy.run_harmony(
            adata.obsm["X_pca"][:, :N_PCS_USE],
            adata.obs,
            BATCH_KEY,
            max_iter_harmony=HARMONY_MAX_ITER,
        )
        # ho.Z_corr 在 harmonypy >= 2.0.0 中已经是 (n_cells, n_pcs) 方向。
        # 输出也是 N_PCS_USE 维——只保留前端有信号维度
        assert ho.Z_corr.shape[1] == N_PCS_USE, (
            f"Harmony Z_corr 维度不符: {ho.Z_corr.shape[1]} != N_PCS_USE({N_PCS_USE})")
        adata.obsm["X_pca_harmony"] = ho.Z_corr
        print(f"obsm['X_pca_harmony'] shape: {adata.obsm['X_pca_harmony'].shape}")

        # --- Harmony 收敛检查（不同版本 harmonypy API 兼容）---
        if hasattr(ho, 'check_convergence') and callable(ho.check_convergence):
            _converged = ho.check_convergence()
        elif hasattr(ho, 'converged'):
            _c = ho.converged
            _converged = _c() if callable(_c) else _c
        else:
            # harmonypy 某些版本没有公开收敛检查端口，默认假设通过
            _converged = True
        if not _converged:
            print(f"⚠️ Harmony 未在 {HARMONY_MAX_ITER} 轮内收敛")
            print(f"  → 考虑: 增大 HARMONY_MAX_ITER 或降低 HARMONY_THETA")
        else:
            print(f"✓ Harmony 收敛（max_iter={HARMONY_MAX_ITER}）")
        _method_status["harmony"] = MethodStatus.SUCCESS
else:
    _method_status["harmony"] = MethodStatus.SKIPPED_BY_USER
    print("Harmony 跳过（HARMONY_ENABLED 为 False）")

### Harmony UMAP（即时查看）

Harmony 在 PCA 空间上迭代校正批次效应。速度快、确定性算法。

**怎么看**：
- 不同样本的细胞是否**混合均匀**？→ 好
- 混合是否过度抹平了生物学差异？→ 警惕过度校正
- 与上方 PCA UMAP 对比：混合改善了，但细胞类型分离是否保留？

如果 Harmony 效果不理想（样本仍分离、或过度混合），可尝试 scVI。

In [ ]:
# Harmony UMAP —— 即时查看批次校正效果
embed_key = "X_pca_harmony"
if embed_key in adata.obsm:
    print(f"\n===== Harmony UMAP =====")
        # N_NEIGHBORS 支持 Scalar-or-Sweep：列表取最后一个，单值直接用
    _k_umap = N_NEIGHBORS if not isinstance(N_NEIGHBORS, list) else N_NEIGHBORS[-1]
    sc.pp.neighbors(adata, use_rep=embed_key,
                    # Harmony 输出 = N_PCS_USE 维，全部使用
                    n_pcs=min(N_PCS_USE, adata.obsm[embed_key].shape[1]),
                    n_neighbors=_k_umap, metric=METRIC,
                    random_state=RANDOM_SEED)
    sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
               random_state=RANDOM_SEED)

    # 保存 UMAP 坐标到专属 key。
    adata.obsm["X_umap_pca_harmony"] = adata.obsm["X_umap"].copy()

    # 三着色出图。
    for colour in colour_columns:
        if colour in adata.obs.columns:
            sc.pl.umap(adata, color=colour, title=f"Harmony - {colour}",
                       frameon=False, save=f"_04_pca_harmony_{colour}.png")
            plt.close("all")

    # doublet score 着色（如果 doublet_score 可用）
    if "doublet_score" in adata.obs.columns and adata.obs["doublet_score"].notna().any():
        sc.pl.umap(adata, color="doublet_score", title=f"Harmony - doublet_score",
                   frameon=False, save="_04_pca_harmony_doublet_score.png")
        plt.close("all")

    print("Harmony UMAP 完成。")
else:
    print("Harmony 嵌入不存在，跳过 UMAP。")

## scVI（从头训练）

scVI（单细胞变分推断）学习一个显式将批次效应建模为干扰变量的潜表示。
需要原始 counts——使用 `adata.layers['counts']`（03 保留的原始计数）。

**快速验证说明**：上方 `SCVI_MAX_EPOCHS` 设得较低以加速管线调试。
生产质量嵌入建议增加到 200-400 epochs。监控训练损失曲线确认收敛。

**为什么至少需要 20 epochs？** scVI 的变分推断需要足够迭代才能收敛到
有意义的潜空间。太少的 epochs 会导致嵌入质量差、下游 UMAP 和聚类
无法反映真实的生物学结构。200-400 epochs 是 scVI 论文和社区的推荐值。

scVI 设置步骤：
1. `setup_anndata`——声明哪些 layer/列存放 counts、batch 等
2. `SCVI(adata)`——实例化模型
3. `.train()`——训练（含早停）
4. `.get_latent_representation()`——提取嵌入到 `obsm["X_scVI"]`

In [ ]:
# scVI：setup_anndata + 训练 + 提取潜表示。
if SCVI_ENABLED:
    if BATCH_KEY not in adata.obs.columns:
        print(f"WARNING: BATCH_KEY='{BATCH_KEY}' 不在 obs 列中，可用列: {list(adata.obs.columns)}；跳过 scVI")
        _method_status["scvi"] = MethodStatus.UNAVAILABLE
    elif adata.obs[BATCH_KEY].nunique() < 2:
        import warnings

        _n_batches = adata.obs[BATCH_KEY].nunique()
        warnings.warn(
            f"检测到 BATCH_KEY='{BATCH_KEY}' 仅包含 {_n_batches} 个唯一值。"
            f"scVI 的 batch_key 指向单值列时，模型将 batch 变量建模为常数，"
            f"batch-specific 参数退化，潜变量不含批次校正信息。"
            f"嵌入结果等同于无批次校正的 VAE。"
            f"建议：将 SCVI_ENABLED 设为 False，改用 PCA 进行嵌入。",
            UserWarning,
        )
        print(f"WARNING: BATCH_KEY='{BATCH_KEY}' 仅 1 个批次，scVI batch_key 无辨识力；自动跳过 scVI")
        _method_status["scvi"] = MethodStatus.UNAVAILABLE
    elif not _counts_valid or _counts_key is None:
        if not _counts_valid:
            print(f"WARNING: scVI 输入校验失败——scVI 跳过。原因: {'; '.join(_counts_failure_reasons)}")
        else:
            print("WARNING: expression_contract 验证失败，无可用的原始 counts——scVI 跳过。")
        _method_status["scvi"] = MethodStatus.UNAVAILABLE
    else:
        print(f"\n===== scVI: max_epochs={SCVI_MAX_EPOCHS}, n_latent={SCVI_N_LATENT} =====")
        import scvi

        _dev = detect_device(prefer=DEVICE, for_method="scvi")
        print(f"scVI 设备: {_dev['device_str']}（{_dev['reason']}）")

        scvi.model.SCVI.setup_anndata(
            adata,
            layer=_counts_key,  # 使用 expression_contract 验证后的 counts 层
            batch_key=BATCH_KEY,
        )

        model = scvi.model.SCVI(
            adata,
            n_layers=SCVI_N_LAYERS,
            n_latent=SCVI_N_LATENT,
            n_hidden=SCVI_N_HIDDEN,
            gene_likelihood=SCVI_GENE_LIKELIHOOD,
            use_layer_norm="both",
            use_batch_norm="none",
        )

        print(f"训练 scVI (max_epochs={SCVI_MAX_EPOCHS})...")
        model.train(
            max_epochs=SCVI_MAX_EPOCHS,
            accelerator=_dev["accelerator"],
            devices=_dev["devices"],
            early_stopping=SCVI_EARLY_STOPPING,
            early_stopping_patience=5,
            plan_kwargs={"lr": 1e-3},
            check_val_every_n_epoch=1,
        )

        adata.obsm["X_scVI"] = model.get_latent_representation()
        print(f"obsm['X_scVI'] shape: {adata.obsm['X_scVI'].shape}")

        # --- scVI 训练收敛诊断 ---
        if hasattr(model, 'history') and "elbo_train" in model.history:
            history_df = model.history["elbo_train"]
            fig, ax = plt.subplots(figsize=(8, 3))
            ax.plot(history_df.index, history_df.values.flatten(), color="steelblue")
            ax.set_xlabel("Epoch"); ax.set_ylabel("ELBO (train)")
            ax.set_title("scVI 训练收敛曲线")
            last10 = history_df.values.flatten()[-10:]
            cv = np.std(last10) / abs(np.mean(last10)) if np.mean(last10) != 0 else 999
            if cv < 0.01:
                print(f"✓ scVI 已收敛（最后 10 epoch CV={cv:.4f}）")
            else:
                print(f"⚠️ scVI 可能未收敛（CV={cv:.4f} > 0.01），建议增大 SCVI_MAX_EPOCHS")
            plt.tight_layout(); plt.show()

        print("scVI 完成。")
        _method_status["scvi"] = MethodStatus.SUCCESS
else:
    _method_status["scvi"] = MethodStatus.SKIPPED_BY_USER
    print("scVI 跳过（SCVI_ENABLED 为 False）")

### scVI UMAP（即时查看）

scVI 通过变分推断学习将批次效应建模为干扰变量的潜表示。
相比 Harmony，能捕捉更复杂的批次-生物学交互。

**怎么看**：
- 样本混合是否优于 PCA 基线？是否优于 Harmony？
- 细胞类型分离是否清晰？
- 训练损失曲线（scVI 自动输出）是否收敛？未收敛 → 增加 `SCVI_MAX_EPOCHS` 重跑

scVI 训练时间较长。如果 UMAP 质量与 Harmony 相当或更差，
且 Harmony 已满足需求，后续可优先使用 Harmony（速度优势）。

In [ ]:
# scVI UMAP —— 即时查看潜空间质量
embed_key = "X_scVI"
if embed_key in adata.obsm:
    print(f"\n===== scVI UMAP =====")
        # N_NEIGHBORS 支持 Scalar-or-Sweep：列表取最后一个，单值直接用
    _k_umap = N_NEIGHBORS if not isinstance(N_NEIGHBORS, list) else N_NEIGHBORS[-1]
    # scVI 潜空间嵌入（use_rep）已经是低维表示（SCVI_N_LATENT 维），
    # scanpy 会使用全部维度构建邻域图——n_pcs 参数仅对 use_rep="X_pca" 生效，此处不传入。
    sc.pp.neighbors(adata, use_rep=embed_key,
                    n_neighbors=_k_umap, metric=METRIC,
                    random_state=RANDOM_SEED)
    sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
               random_state=RANDOM_SEED)

    # 保存 UMAP 坐标到专属 key。
    adata.obsm["X_umap_scVI"] = adata.obsm["X_umap"].copy()

    # 三着色出图。
    for colour in colour_columns:
        if colour in adata.obs.columns:
            sc.pl.umap(adata, color=colour, title=f"scVI - {colour}",
                       frameon=False, save=f"_04_scVI_{colour}.png")
            plt.close("all")

    # doublet score 着色（如果 doublet_score 可用）
    if "doublet_score" in adata.obs.columns and adata.obs["doublet_score"].notna().any():
        sc.pl.umap(adata, color="doublet_score", title=f"scVI - doublet_score",
                   frameon=False, save="_04_scVI_doublet_score.png")
        plt.close("all")

    print("scVI UMAP 完成。")
else:
    print("scVI 嵌入不存在，跳过 UMAP。")

## （可选）cellxgene_census 预训练 scVI

使用 CZ CELLxGENE Census 预训练 scVI 模型对当前数据集进行嵌入。
该预训练模型将新细胞映射到从 CELLxGENE 语料库数百万细胞中学到的
潜空间中——无需本地训练。

**前置条件**：Census 必须托管了该组织（示例数据中为 gastric mucosa）
的 scVI 模型。可在 cellxgene.cziscience.com 查询可用性。

启用后，下方 cell 将：
1. 从 Census 下载预训练模型。
2. 通过 mygene 对齐基因符号。
3. 调用 `prepare_query_anndata` 和 `load_query_data`。
4. 提取潜表示到 `obsm["X_scVI_census"]`。

该 cell 默认**被注释掉**，因为 Census 作为重量级依赖增加安装负担，
且仅在组织存在优质模型时才有效。

In [ ]:
# # === cellxgene_census 预训练 scVI（已注释） ===
# # 前置条件：Census 存在该组织的预训练 scVI 模型。
# # 启用此 cell：取消下方注释，然后在 PARAMS 中添加
# # CENSUS_ENABLED = True。
#
# # import cellxgene_census
# # census = cellxgene_census.open_soma(census_version="latest")
# # model = cellxgene_census.download_source_h5ad(
# #     ORGANISM, layer="scvi", tissue=TISSUE,
# # )
# # # 基因对齐（参见 legacy-GCPL/04_dimensionality_reduction.ipynb）
# # adata.obsm["X_scVI_census"] = model.get_latent_representation(adata)
# # print("cellxgene_census scVI 完成。")
#
# print("cellxgene_census 预训练 scVI cell 已注释。"
#       "Census 存在该组织模型时取消注释。")

## （可选）scANVI 标签迁移嵌入

scANVI（single-cell ANnotation Variational Inference，单细胞注释变分推断）
在 scVI 基础上扩展了细胞类型分类头。其产出的潜空间兼具批次校正与
细胞类型感知能力——当存在带标注的参考数据集且细胞类型在批次间
保守时尤为有用。

**前置条件**：`adata.obs` 中必须包含可用的细胞类型标签列。
本 cell 自动检测常见标签列名：
`cell_type_original_*`、`Celltypes_global`、`cell_type`、`label`。
若均未找到，则优雅跳过 scANVI 并输出日志提示。

In [ ]:
# scANVI：scVI 的半监督变体，利用细胞类型标签。
# 需显式设置 SCANVI_LABEL_KEY 指定标签列名（不再自动搜索）。
if SCANVI_ENABLED:
    print("\n===== scANVI =====")

    # scANVI 依赖 scVI 的潜表示（X_scVI）；若 EMBEDDING_METHODS 含 scanvi 但不含 scvi，
    # scANVI 会独立训练 scVI 模型，但 scVI UMAP 不可用于对比。建议同时启用 scvi。
    if not SCVI_ENABLED:
        print("⚠️ SCANVI_ENABLED=True 但 SCVI_ENABLED=False。")
        print("   scANVI 将独立训练 scVI 模型，但无独立 scVI UMAP 可供对比。")
        print("   建议同时设置 SCVI_ENABLED = True。")

    if BATCH_KEY not in adata.obs.columns:
        print(f"WARNING: BATCH_KEY='{BATCH_KEY}' 不在 obs 列中，无法执行 batch 相关检查；自动跳过 scANVI")
        _method_status["scanvi"] = MethodStatus.UNAVAILABLE
    elif adata.obs[BATCH_KEY].nunique() < 2:
        import warnings

        _n_batches = adata.obs[BATCH_KEY].nunique()
        warnings.warn(
            f"检测到 BATCH_KEY='{BATCH_KEY}' 仅包含 {_n_batches} 个唯一值。"
            f"scANVI 同样将 batch_key 用于 batch-specific 参数——单 batch 下参数退化，"
            f"加上标签引导可能过度依赖标签信息而忽略数据本身的变异结构。"
            f"建议：将 SCANVI_ENABLED 设为 False。",
            UserWarning,
        )
        print(f"WARNING: BATCH_KEY='{BATCH_KEY}' 仅 1 个批次；自动跳过 scANVI")
        _method_status["scanvi"] = MethodStatus.UNAVAILABLE
    elif not _counts_valid or _counts_key is None:
        print("scANVI 跳过: expression_contract 验证失败，无可用的原始 counts。")
        _method_status["scanvi"] = MethodStatus.UNAVAILABLE
    elif SCANVI_LABEL_KEY is None:
        print("scANVI 跳过: SCANVI_LABEL_KEY 未设置（不再自动搜索标签列）。")
        print("  启用 scANVI: 将 SCANVI_LABEL_KEY 设置为 adata.obs 中的有效标签列名。")
        _method_status["scanvi"] = MethodStatus.SKIPPED_BY_USER
    elif SCANVI_LABEL_KEY not in adata.obs.columns:
        print(f"scANVI 跳过: SCANVI_LABEL_KEY='{SCANVI_LABEL_KEY}' 不在 obs 列中。")
        print(f"  可用 obs 列: {list(adata.obs.columns)[:10]} ...")
        _method_status["scanvi"] = MethodStatus.UNAVAILABLE
    else:
        label_col = SCANVI_LABEL_KEY
        n_labels = adata.obs[label_col].nunique()
        n_nan = adata.obs[label_col].isna().sum()
        print(f"标签列: '{label_col}' ({n_labels} unique, {n_nan} NaN)")

        if n_labels < 2:
            print(f"scANVI 跳过: '{label_col}' 的 unique 非 NaN 值 < 2。")
            _method_status["scanvi"] = MethodStatus.UNAVAILABLE
        else:
            import scvi
            from scvi.model import SCANVI

            _dev = detect_device(prefer=DEVICE, for_method="scanvi")
            print(f"scANVI 设备: {_dev['device_str']}（{_dev['reason']}）")

            scvi.model.SCVI.setup_anndata(
                adata, layer=_counts_key, batch_key=BATCH_KEY,
                labels_key=label_col,
            )
            scvi_model = scvi.model.SCVI(
                adata, n_layers=SCVI_N_LAYERS, n_latent=SCVI_N_LATENT,
                n_hidden=SCVI_N_HIDDEN,
                gene_likelihood=SCVI_GENE_LIKELIHOOD, use_layer_norm="both",
                use_batch_norm="none",
            )
            scvi_model.train(max_epochs=SCVI_MAX_EPOCHS,
                             accelerator=_dev["accelerator"], devices=_dev["devices"],
                             early_stopping=SCVI_EARLY_STOPPING,
                             early_stopping_patience=5,
                             plan_kwargs={"lr": 1e-3})

            scanvi_model = SCANVI.from_scvi_model(
                scvi_model,
                unlabeled_category="Unknown",
                labels_key=label_col,
            )
            n_ep = SCANVI_N_EPOCHS
            print(f"训练 scANVI (max_epochs={n_ep})...")
            scanvi_model.train(
                max_epochs=n_ep,
                accelerator=_dev["accelerator"],
                devices=_dev["devices"],
                early_stopping=SCVI_EARLY_STOPPING,
                early_stopping_patience=3,
            )

            adata.obsm["X_scANVI"] = scanvi_model.get_latent_representation()
            print(f"obsm['X_scANVI'] shape: {adata.obsm['X_scANVI'].shape}")
            print("scANVI 完成。")
            _method_status["scanvi"] = MethodStatus.SUCCESS
else:
    _method_status["scanvi"] = MethodStatus.SKIPPED_BY_USER
    print("scANVI 跳过（SCANVI_ENABLED 为 False）")

### scANVI UMAP（即时查看）

scANVI 在 scVI 基础上融入细胞类型标签，产出的潜空间**同时感知批次校正与细胞类型**。

**怎么看**：
- 与上方 scVI UMAP 对比：细胞类型分离是否更清晰？
- 样本混合是否维持？
- 如果标签质量差或覆盖不全，scANVI 可能不如纯 scVI——对比后择优

In [ ]:
# scANVI UMAP —— 即时查看标签感知潜空间（仅当 scANVI 已运行）
embed_key = "X_scANVI"
if embed_key in adata.obsm:
    print(f"\n===== scANVI UMAP =====")
        # N_NEIGHBORS 支持 Scalar-or-Sweep：列表取最后一个，单值直接用
    _k_umap = N_NEIGHBORS if not isinstance(N_NEIGHBORS, list) else N_NEIGHBORS[-1]
    # scANVI 潜空间嵌入（use_rep）已经是低维表示（SCVI_N_LATENT 维），
    # scanpy 会使用全部维度构建邻域图——n_pcs 参数仅对 use_rep="X_pca" 生效，此处不传入。
    sc.pp.neighbors(adata, use_rep=embed_key,
                    n_neighbors=_k_umap, metric=METRIC,
                    random_state=RANDOM_SEED)
    sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
               random_state=RANDOM_SEED)

    # 保存 UMAP 坐标到专属 key。
    adata.obsm["X_umap_scANVI"] = adata.obsm["X_umap"].copy()

    # 三着色出图。
    for colour in colour_columns:
        if colour in adata.obs.columns:
            sc.pl.umap(adata, color=colour, title=f"scANVI - {colour}",
                       frameon=False, save=f"_04_scANVI_{colour}.png")
            plt.close("all")

    # doublet score 着色（如果 doublet_score 可用）
    if "doublet_score" in adata.obs.columns and adata.obs["doublet_score"].notna().any():
        sc.pl.umap(adata, color="doublet_score", title=f"scANVI - doublet_score",
                   frameon=False, save="_04_scANVI_doublet_score.png")
        plt.close("all")

    print("scANVI UMAP 完成。")
else:
    print("scANVI 嵌入不存在（未运行或无可用的细胞类型标签），跳过 UMAP。")

## scCRAFT（可选嵌入方法，anchor-free 批次整合）

scCRAFT 是一种 anchor-free 的批次校正整合方法（VAE + 判别器 + 部分拓扑结构）。
与 scVI 类似从原始 counts 学习潜空间，但用对抗训练增强批次混合。

**安装**（非 PyPI 包，需从 GitHub 源码安装）:
```bash
git clone https://github.com/ch2343/scCRAFT && cd scCRAFT && pip install .
```

**重要**: scCRAFT 内部会对传入的 AnnData 做 normalize + HVG 子集化（会修改对象）。因此本 notebook 在**独立的 counts 副本**上运行 scCRAFT，绝不触碰主 adata——结果通过 obs_names 对齐写回 `adata.obsm["X_scCRAFT"]`，主 adata 的 X / layers / 其他 obsm 完全不受影响。

In [ ]:
# scCRAFT：在独立 counts 副本上训练，结果对齐写回主 adata。
# 关键隔离：scCRAFT 内部会 normalize + log1p + 子集化到 HVG，
# 这些突变必须在独立副本上完成，绝不触碰主 adata。
if SCCRAFT_ENABLED:
    if BATCH_KEY not in adata.obs.columns:
        print(f"WARNING: BATCH_KEY='{BATCH_KEY}' 不在 obs 列中；跳过 scCRAFT")
        _method_status["sccraft"] = MethodStatus.UNAVAILABLE
    elif not _counts_valid or _counts_key is None:
        print("WARNING: expression_contract 验证失败，无可用的原始 counts——scCRAFT 跳过。")
        _method_status["sccraft"] = MethodStatus.UNAVAILABLE
    else:
        try:
            import scCRAFT
            from scCRAFT.model import (
                multi_resolution_cluster,
                train_integration_model,
                obtain_embeddings,
            )
            _sccraft_available = True
        except ImportError:
            print("scCRAFT 未安装，跳过。")
            print("   安装: git clone https://github.com/ch2343/scCRAFT && cd scCRAFT && pip install .")
            _sccraft_available = False
            _method_status["sccraft"] = MethodStatus.UNAVAILABLE

        if _sccraft_available:
            print(f"\n===== scCRAFT: epochs={SCCRAFT_EPOCHS}, d_coef={SCCRAFT_D_COEF}, kl_coef={SCCRAFT_KL_COEF} =====")
            import anndata as _ad

            # 构建独立的 counts AnnData——绝不 mutate 主 adata
            # scCRAFT 内部会 normalize + log1p + HVG 子集化，必须在副本上做
            _sccraft_adata = _ad.AnnData(
                X=adata.layers[_counts_key].copy(),
                obs=adata.obs[[BATCH_KEY]].copy(),
                var=adata.var[[]].copy(),
            )
            _sccraft_adata.obs_names = adata.obs_names.copy()
            _sccraft_adata.var_names = adata.var_names.copy()
            print(f"  独立副本: {_sccraft_adata.n_obs} cells x {_sccraft_adata.n_vars} genes")

            # scCRAFT 标准预处理（在副本上——不影响主 adata）
            _sccraft_adata.raw = _sccraft_adata.copy()
            sc.pp.filter_genes(_sccraft_adata, min_cells=5)
            sc.pp.normalize_per_cell(_sccraft_adata, counts_per_cell_after=1e4)
            sc.pp.log1p(_sccraft_adata)
            sc.pp.highly_variable_genes(_sccraft_adata, n_top_genes=SCCRAFT_N_TOP_GENES,
                                        batch_key=BATCH_KEY)
            _sccraft_adata = _sccraft_adata[:, _sccraft_adata.var["highly_variable"]].copy()
            print(f"  HVG 子集后: {_sccraft_adata.n_vars} genes")

            try:
                # scCRAFT 训练流程
                multi_resolution_cluster(_sccraft_adata, resolution1=SCCRAFT_RESOLUTION,
                                         method=SCCRAFT_CLUSTER_METHOD)
                # 注：scCRAFT 内部硬编码 CPU（self.device='cpu'），DEVICE 参数对其无效。
                # Mac/Linux 均走 CPU，跨平台一致。
                _dev_sccraft = detect_device(prefer=DEVICE, for_method="sccraft")
                print(f"scCRAFT 设备: {_dev_sccraft['device_str']}（{_dev_sccraft['reason']}）")
                _vae = train_integration_model(
                    _sccraft_adata, batch_key=BATCH_KEY,
                    epochs=SCCRAFT_EPOCHS, d_coef=SCCRAFT_D_COEF,
                    kl_coef=SCCRAFT_KL_COEF, warmup_epoch=SCCRAFT_WARMUP_EPOCH,
                )
                obtain_embeddings(_sccraft_adata, _vae)

                # 对齐写回主 adata（按 obs_names 保证行顺序一致）
                if "X_scCRAFT" in _sccraft_adata.obsm:
                    _embed_df = pd.DataFrame(
                        _sccraft_adata.obsm["X_scCRAFT"],
                        index=_sccraft_adata.obs_names,
                    )
                    # 按主 adata 的 obs_names 顺序对齐
                    _embed_aligned = _embed_df.reindex(adata.obs_names).values
                    adata.obsm["X_scCRAFT"] = _embed_aligned.astype(np.float32)
                    print(f"  obsm['X_scCRAFT'] shape: {adata.obsm['X_scCRAFT'].shape}")
                    # 验证无 NaN（对齐失败会产生 NaN）
                    _n_nan = np.isnan(adata.obsm["X_scCRAFT"]).any(axis=1).sum()
                    if _n_nan > 0:
                        print(f"  ⚠️ {_n_nan} 个细胞的 scCRAFT 嵌入为 NaN（obs_names 对齐失败）")
                        print(f"  → 删除 X_scCRAFT（避免含 NaN 数据流入下游 UMAP/指标/推荐）")
                        del adata.obsm["X_scCRAFT"]
                        _method_status["sccraft"] = MethodStatus.FAILED
                    else:
                        _method_status["sccraft"] = MethodStatus.SUCCESS
                else:
                    print("WARNING: scCRAFT 未产出 X_scCRAFT，检查训练是否成功")
                    _method_status["sccraft"] = MethodStatus.FAILED
            except Exception as e:
                print(f"WARNING: scCRAFT 训练失败: {e}")
                import traceback
                traceback.print_exc()
                _method_status["sccraft"] = MethodStatus.FAILED
            finally:
                # 清理副本释放内存
                del _sccraft_adata
                import gc as _gc
                _gc.collect()
            print("scCRAFT 完成。")
else:
    _method_status["sccraft"] = MethodStatus.SKIPPED_BY_USER
    print("scCRAFT 跳过（SCCRAFT_ENABLED 为 False）")

In [ ]:
# scCRAFT UMAP —— 即时查看潜空间质量
embed_key = "X_scCRAFT"
if embed_key in adata.obsm:
    print(f"\n===== scCRAFT UMAP =====")
    _sc_dim = adata.obsm[embed_key].shape[1]
    # 独立 UMAP 用 N_NEIGHBORS 仅作可视化初判；定量整合指标见后文整合指标对比 cell（用自适应 k）
    sc.pp.neighbors(adata, use_rep=embed_key,
                    n_pcs=min(_sc_dim, adata.obsm[embed_key].shape[1]),
                    n_neighbors=N_NEIGHBORS, random_state=RANDOM_SEED)
    sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
               random_state=RANDOM_SEED)

    # 保存 UMAP 坐标到专属 key。
    adata.obsm["X_umap_scCRAFT"] = adata.obsm["X_umap"].copy()

    # 三着色出图：样本、批次、细胞类型（如有）。
    for colour in colour_columns:
        if colour in adata.obs.columns:
            sc.pl.umap(adata, color=colour, title=f"scCRAFT - {colour}",
                       frameon=False, save=f"_04_scCRAFT_{colour}.png")
            plt.close("all")

    # doublet score 着色（如果 doublet_score 可用）
    if "doublet_score" in adata.obs.columns and adata.obs["doublet_score"].notna().any():
        sc.pl.umap(adata, color="doublet_score", title=f"scCRAFT - doublet_score",
                   frameon=False, save="_04_scCRAFT_doublet_score.png")
        plt.close("all")

    print("scCRAFT UMAP 完成。")
else:
    print("scCRAFT UMAP 跳过（X_scCRAFT 不存在）")

In [ ]:
# --- Scalar-or-Sweep：N_NEIGHBORS ---
# 不同 k 值影响 UMAP 局部-全局结构平衡。
# 列表模式：对每个 k 独立计算 neighbors + UMAP，grid 对比形态差异。
# 单值模式：跳过 sweep，下游直接使用主流程中已计算的 neighbors。
_k_values = N_NEIGHBORS if isinstance(N_NEIGHBORS, list) else [N_NEIGHBORS]

if len(_k_values) > 1:
    # 选择 sweep 用的嵌入：优先 X_pca_harmony，其次 X_pca
    sweep_rep = None
    for candidate in ["X_pca_harmony", "X_pca"]:
        if candidate in adata.obsm:
            sweep_rep = candidate
            break
    if sweep_rep is None:
        print("N_NEIGHBORS sweep 跳过：无可用的 obsm 嵌入。")
    else:
        print(f"N_NEIGHBORS sweep: {_k_values} on {sweep_rep}")

        n_cols = min(len(_k_values), 4)
        n_rows = (len(_k_values) + n_cols - 1) // n_cols
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
        # 统一 flatten：单 subplot 时 axes 不是 ndarray
        if n_rows == 1 and n_cols == 1:
            axes_flat = [axes]
        else:
            axes_flat = axes.flatten()

        for i, k in enumerate(_k_values):
            sc.pp.neighbors(adata, use_rep=sweep_rep, n_neighbors=k,
                            metric=METRIC, random_state=RANDOM_SEED)
            sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
                       random_state=RANDOM_SEED)
            adata.obsm[f"X_umap_k{k}"] = adata.obsm["X_umap"].copy()
            color_col = BATCH_KEY if BATCH_KEY in adata.obs.columns else colour_columns[0]
            sc.pl.umap(adata, color=color_col, ax=axes_flat[i], show=False,
                       title=f"k={k}")

        # 隐藏多余 subplot
        for j in range(i + 1, len(axes_flat)):
            axes_flat[j].set_visible(False)

        plt.suptitle(f"N_NEIGHBORS sweep（{sweep_rep}，不同 k 下 UMAP 形态）")
        plt.tight_layout(); plt.show()

        # 恢复为列表最后一个 k 的 neighbors + UMAP
        sc.pp.neighbors(adata, use_rep=sweep_rep, n_neighbors=_k_values[-1],
                        metric=METRIC, random_state=RANDOM_SEED)
        sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
                   random_state=RANDOM_SEED)
        print(f"N_NEIGHBORS sweep 完成。最终使用 k={_k_values[-1]} 继续下游。")
else:
    print(f"N_NEIGHBORS 单值模式 (k={_k_values[0]})，跳过 sweep。")

In [ ]:
# --- Scalar-or-Sweep：HARMONY_THETA ---
# 不同 theta 值控制 Harmony 批次校正强度。
# theta 越大 = 校正越强（样本混合越好，但可能过度抹平生物学差异）。
# 列表模式：对每个 theta 重新运行 Harmony + UMAP，grid 对比。
# 单值模式：跳过 sweep。
_t_values = HARMONY_THETA if isinstance(HARMONY_THETA, list) else [HARMONY_THETA]

if len(_t_values) > 1 and "X_pca" in adata.obsm:
    print(f"HARMONY_THETA sweep: {_t_values}")
    import harmonypy

    n_cols = min(len(_t_values), 4)
    n_rows = (len(_t_values) + n_cols - 1) // n_cols
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(5 * n_cols, 5 * n_rows))
    if n_rows == 1 and n_cols == 1:
        axes_flat = [axes]
    else:
        axes_flat = axes.flatten()

    _use_k = N_NEIGHBORS if not isinstance(N_NEIGHBORS, list) else _k_values[-1]

    for i, theta in enumerate(_t_values):
        # 只喂前 N_PCS_USE 个 PC——与主 Harmony cell 保持一致
        ho = harmonypy.run_harmony(
            adata.obsm["X_pca"][:, :N_PCS_USE], adata.obs, BATCH_KEY,
            theta=theta, max_iter_harmony=HARMONY_MAX_ITER,
        )
        adata.obsm[f"X_pca_harmony_t{theta}"] = ho.Z_corr

        sc.pp.neighbors(adata, use_rep=f"X_pca_harmony_t{theta}",
                        n_neighbors=_use_k, metric=METRIC,
                        n_pcs=N_PCS_USE, random_state=RANDOM_SEED)
        sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
                   random_state=RANDOM_SEED)

        color_col = BATCH_KEY if BATCH_KEY in adata.obs.columns else colour_columns[0]
        sc.pl.umap(adata, color=color_col, ax=axes_flat[i], show=False,
                   title=f"theta={theta}")

    for j in range(i + 1, len(axes_flat)):
        axes_flat[j].set_visible(False)

    plt.suptitle("HARMONY_THETA sweep（不同 theta 下批次混合 vs 生物学保留）")
    plt.tight_layout(); plt.show()

    # 恢复为主 HARMONY_THETA（列表最后一个值）的嵌入
    adata.obsm["X_pca_harmony"] = adata.obsm[f"X_pca_harmony_t{_t_values[-1]}"].copy()
    sc.pp.neighbors(adata, use_rep="X_pca_harmony", n_neighbors=_use_k,
                    metric=METRIC, n_pcs=N_PCS_USE, random_state=RANDOM_SEED)
    sc.tl.umap(adata, min_dist=UMAP_MIN_DIST, spread=UMAP_SPREAD,
               random_state=RANDOM_SEED)
    print(f"HARMONY_THETA sweep 完成。最终使用 theta={_t_values[-1]}。")
elif HARMONY_ENABLED and "X_pca" in adata.obsm:
    print(f"HARMONY_THETA 单值模式 (theta={_t_values[0]})，跳过 sweep。")
else:
    print("Harmony 未启用或 PCA 未计算，跳过 HARMONY_THETA sweep。")

In [ ]:
# === Marker 基因 UMAP 叠印（评估嵌入空间的生物信号保留）===
# 使用最后跑完的嵌入的 UMAP
_last_umap = None
for _k in ["X_umap_scVI", "X_umap_pca_harmony", "X_umap_scANVI", "X_umap_pca"]:
    if _k in adata.obsm:
        _last_umap = _k
        break

if _last_umap:
    # 临时设为默认 UMAP（sc.pl.umap 使用 X_umap）
    _backup_umap = adata.obsm.get("X_umap", None)
    adata.obsm["X_umap"] = adata.obsm[_last_umap]
    
    _avail = [g for g in MARKER_GENES_UMAP if g in adata.var_names]
    if _avail:
        print(f"Marker 基因 UMAP 叠印（{_last_umap} 空间，{len(_avail)} 基因）")
        sc.pl.umap(adata, color=_avail, ncols=4, frameon=False,
                   vmin=0, vmax="p99", show=True,
                   save="_04_marker_genes.png")
        
        print("  解读: 各 marker 应在 UMAP 上形成清晰分离区域")
        print("  如 EPCAM+/PTPRC+ 混在一起 -> 嵌入过校正，考虑降低 Harmony theta 或换 scVI")
    
    # 恢复原有 UMAP
    if _backup_umap is not None:
        adata.obsm["X_umap"] = _backup_umap
    else:
        del adata.obsm["X_umap"]
else:
    print("无可用 UMAP 嵌入，跳过 marker UMAP 叠印")


## 整合指标对比（佐证）

上方已对每个嵌入单独出 UMAP 图供目视判断。本 cell 用定量指标做**补充佐证**——
**UMAP 目测为主决策，指标辅助**。

直接遍历所有已有嵌入，对每个嵌入：
1. 拷贝 adata 避免相互干扰
2. 计算邻居图 + UMAP
3. 对每个嵌入做内联 silhouette_score 计算（批次混合 + 细胞类型保留），代码显式可见——无 import，无黑盒
4. 收集结果到对比表

**没有回调、没有 `sweep()`**——学生打开 notebook 能逐行看懂每一步。

指标（数据允许时计算）：
- **silhouette_batch**——越低 = 批次混合越好
- **silhouette_celltype**——越高 = 生物学信号保留越好
- **scib_available**——1.0 表示 scib-metrics 已安装，0.0 表示未安装

对比表写入 `results/figures/sweep_04/sweep_report.md`。
PI 对照 UMAP 图和指标表决定选用哪个嵌入。

In [ ]:
# 显式 for 循环：遍历各嵌入，直接计算 neighbors + UMAP + 整合指标。
# 每一步都是标准 scanpy 操作——学生可以逐行阅读和理解，无需学习回调模型。
print("\n===== 显式遍历嵌入 + 整合指标 =====\n")

import pandas as pd

# 只遍历已计算且存在于 obsm 中的嵌入
use_reps = [k for k in ["X_pca", "X_pca_harmony", "X_scVI", "X_scANVI", "X_scCRAFT"]
            if k in adata.obsm]
if not use_reps:
    print("警告: 未在 obsm 中找到任何嵌入。跳过。")
else:
    print(f"遍历 {len(use_reps)} 个嵌入: {use_reps}\n")

    # 基准 k 值（用于自适应缩放）
    _base_k = N_NEIGHBORS if not isinstance(N_NEIGHBORS, list) else N_NEIGHBORS[-1]

    results = []
    for rep in use_reps:
        print(f"--- {rep} ---")

        # obsm 存在性防御——防止某方法在 EMBEDDING_METHODS 中声明
        # 但该方法的嵌入 cell 被跳过/失败导致 obsm 缺失
        if rep not in adata.obsm:
            print(f"跳过 {rep}：无 {rep}（该方法未运行或未产出嵌入）")
            continue

        # 拷贝 AnnData 避免不同嵌入之间干扰（邻居图、UMAP 坐标不共用）
        adata_copy = adata.copy()

        # 维度自适应邻居数：低维嵌入用较少邻居，高维嵌入适当增加
        embed_dim = adata_copy.obsm[rep].shape[1]
        _adaptive_k = max(10, int(_base_k * min(1.0, embed_dim / 30)))

        # PCA/Harmony 嵌入空间用 N_PCS_USE 截断；
        # scVI/scANVI/scCRAFT 潜空间本身低维，全部使用
        if rep.startswith("X_sc"):
            n_dim = embed_dim  # 低维潜嵌入用全部维度
        else:
            n_dim = min(N_PCS_USE, embed_dim)

        # 在嵌入空间计算邻居图 + UMAP
        sc.pp.neighbors(adata_copy, use_rep=rep, n_neighbors=_adaptive_k,
                        n_pcs=n_dim, random_state=RANDOM_SEED)
        sc.tl.umap(adata_copy, random_state=RANDOM_SEED)

        # === 整合指标内联计算（原 scorers.integration_metrics，现留 notebook cell 可见可调）===
        # 计算原理：在嵌入空间上用 silhouette_score 评估批次混合（越低越好）与细胞类型保留（越高越好）
        _m = {}
        _x = adata_copy.obsm[rep]  # 嵌入坐标矩阵 (n_cells, n_dims)

        # 1. 批次混合指标（silhouette_batch）
        #    同一批次内 cells 的嵌入相似度越高 → silhouette_batch 越高（批次分离明显 → 校正不足）
        #    值越低 = 批次混合越好，理想值趋近 0 或负
        if BATCH_KEY in adata_copy.obs.columns:
            _b_labels = adata_copy.obs[BATCH_KEY].astype(str)
            _bm = adata_copy.obs[BATCH_KEY].notna()
            if _bm.sum() >= 2 and _b_labels[_bm].nunique() >= 2:
                try:
                    _m["silhouette_batch"] = float(
                        silhouette_score(_x[_bm], _b_labels[_bm])
                    )
                except Exception:
                    pass

        # 2. 生物学信号保留指标（silhouette_celltype）
        #    同一细胞类型内 cells 的嵌入相似度越高 → silhouette_celltype 越高（生物学分离好）
        #    值越高 = 生物学信号保留越好，理想值 > 0.3
        _ct_key = None
        for _cand in ["cell_type", "cell_type_final_v1", "label", "cell_type_original"]:
            for _col in adata_copy.obs.columns:
                if _cand in _col:
                    _ct_key = _col
                    break
            if _ct_key:
                break
        if _ct_key:
            _ct_labels = adata_copy.obs[_ct_key].astype(str)
            _cm = adata_copy.obs[_ct_key].notna()
            if _cm.sum() >= 2 and _ct_labels[_cm].nunique() >= 2:
                try:
                    _m["silhouette_celltype"] = float(
                        silhouette_score(_x[_cm], _ct_labels[_cm])
                    )
                except Exception:
                    pass

        # 3. scib-metrics 可用性标记（1.0=已安装，可用于更全面的整合评估；0.0=未安装）
        try:
            import scib_metrics  # noqa: F401
            _m["scib_available"] = 1.0
        except ImportError:
            _m["scib_available"] = 0.0

        results.append({"use_rep": rep, **_m})

        # 输出当前嵌入的指标摘要
        metrics_str = ", ".join(f"{k}={v:.4f}" for k, v in _m.items()
                                if isinstance(v, float) and not np.isnan(v))
        print(f"  指标: {metrics_str}")

    # 收集为 DataFrame 对比表
    sweep_df = pd.DataFrame(results)
    os.makedirs("results/figures/sweep_04", exist_ok=True)

    # 写 Markdown 报告（无依赖，纯手写表格）
    lines = ["# 04 嵌入对比报告\n",
             f"**{len(use_reps)} 个嵌入** 已评估。\n",
             "## 指标表\n"]
    lines.append("| " + " | ".join(sweep_df.columns) + " |")
    lines.append("|" + "|".join(" --- " for _ in sweep_df.columns) + "|")
    for _, row in sweep_df.iterrows():
        vals = []
        for col in sweep_df.columns:
            v = row[col]
            if isinstance(v, float):
                vals.append(f"{v:.4f}" if not np.isnan(v) else "N/A")
            else:
                vals.append(str(v))
        lines.append("| " + " | ".join(vals) + " |")
    with open("results/figures/sweep_04/sweep_report.md", "w") as f:
        f.write("\n".join(lines) + "\n")

    # 展示对比表
    print("\n整合指标对比表:")
    try:
        from IPython.display import display as ipy_display
        ipy_display(sweep_df)
    except ImportError:
        print(sweep_df)

    print("\n对比报告: results/figures/sweep_04/sweep_report.md")
    adata.uns["04_sweep_v1"] = {
        "embeddings_swept": use_reps,
        "scorer": "inline_silhouette_batch_celltype",
        "report_dir": "results/figures/sweep_04",
        "timestamp": datetime.datetime.now().isoformat(),
    }

In [ ]:
# --- 过校正检测：检查 batch 校正是否误删了真实生物学差异 ---
# 原理：好的整合 = 同类型跨 batch 混合（好）+ 不同类型仍然分开（保留）
_ct_col = None
for c in ["cell_type_original_Nowicki_2023_v1", "Celltypes_global", "cell_type", "cell_type_final_v1"]:
    if c in adata.obs.columns and adata.obs[c].notna().sum() > 100:
        _ct_col = c
        break

if _ct_col:
    from sklearn.neighbors import NearestNeighbors

    print(f"===== 过校正检测（使用 {_ct_col}）=====")
    # 用最终选定的嵌入（优先 X_pca_harmony，其次 X_pca）
    _embed = adata.obsm.get("X_pca_harmony", adata.obsm.get("X_pca"))
    nn = NearestNeighbors(n_neighbors=50, metric="cosine").fit(_embed)
    _, indices = nn.kneighbors(_embed)

    labels = adata.obs[_ct_col].values
    batches = adata.obs[BATCH_KEY].values

    # 每个细胞的 50 近邻中：同类型不同 batch 的比例（越高越好）
    batch_mix_scores = []
    # 每个细胞的 50 近邻中：不同类型的比例（越低越好）
    type_mix_scores = []

    for i in range(len(labels)):
        neighbors = indices[i, 1:]  # 排除自身
        same_type = labels[neighbors] == labels[i]
        diff_batch = batches[neighbors] != batches[i]
        batch_mix_scores.append((same_type & diff_batch).mean())
        type_mix_scores.append((~same_type).mean())

    batch_mixing = np.mean(batch_mix_scores)
    type_mixing = np.mean(type_mix_scores)
    print(f"  Batch mixing within cell type: {batch_mixing:.3f}（越高越好，说明同类型跨 batch 混合良好）")
    print(f"  Type mixing across neighbors: {type_mixing:.3f}（越低越好，说明不同类型仍然分离）")
    if type_mixing > 0.3:
        print(f"  ⚠️ type_mixing > 0.3：可能存在过校正，不同细胞类型被错误混合")
    elif batch_mixing < 0.1:
        print(f"  ⚠️ batch_mixing < 0.1：batch 效应可能未被充分校正")
    else:
        print(f"  ✓ 整合质量良好：batch 混合充分且类型分离保持")
else:
    print("跳过过校正检测：无可用的 cell_type 列")

In [ ]:
# === 嵌入方法选择建议 ===
# 以下为基于整合指标的自动化建议。**指标是佐证，不是判据**——
# 最终决策仍由 PI 基于 UMAP 目测做出。指标可能被异常细胞/标签
# 噪声干扰，不能替代人对嵌入空间生物学合理性的判断。
print("===== 嵌入方法选择建议 =====")
_available_reps = [k for k in ["X_pca", "X_pca_harmony", "X_scVI", "X_scANVI", "X_scCRAFT"]
                   if k in adata.obsm]
print(f"可用嵌入: {_available_reps}")
print()

_results_list = globals().get("results", None)
if _results_list is not None and isinstance(_results_list, list) and len(_results_list) > 1:
    _df = pd.DataFrame(_results_list)
    if "silhouette_batch" in _df.columns and "silhouette_celltype" in _df.columns:
        _best_batch = _df.loc[_df["silhouette_batch"].idxmin(), "use_rep"]
        _best_bio = _df.loc[_df["silhouette_celltype"].idxmax(), "use_rep"]
        print(f"  批次混合最佳: {_best_batch} (silhouette_batch 最低)")
        print(f"  生物学保留最佳: {_best_bio} (silhouette_celltype 最高)")
        if _best_batch == _best_bio:
            print(f"  → 推荐: {_best_batch}（批次混合 + 生物学保留兼优）")
        else:
            print(f"  → 需 PI 目测 UMAP 权衡:")
            print(f"    {_best_batch}（混合好但可能过校正）")
            print(f"    {_best_bio}（分离清晰但可能欠校正）")
        print()
        # 输出可直接复制到 stage 05 的推荐参数
        _recommend = _best_bio if _best_bio != "X_pca" else _best_batch
        print(f"stage 05 推荐设置:")
        print(f'  USE_REP = "{_recommend}"')
    else:
        print("整合指标缺少 silhouette 列——无法生成建议。请目测 UMAP 决定。")
elif len(_available_reps) == 1:
    print(f"仅一种嵌入可用，默认使用: {_available_reps[0]}")
else:
    print("无整合指标结果（可能只跑了 PCA 且无细胞类型标签）")
    print("→ 请目测 UMAP 决定")

In [ ]:
# === 研究者显式选择 selected_embedding（UX-1 决策 cell）===
# 本 cell 是研究者在全面审查嵌入结果后的关键决定。
# 该决定决定 05 聚类使用哪个嵌入空间，因此必须明确且手动完成——
# 不自动赋值、不指标推荐替代。
_decision_completed = False

# 研究者修改下方赋值以选择嵌入（示例: SELECTED_EMBEDDING = "X_scVI"）
# SELECTED_EMBEDDING = None  # ← 改为实际选择的 obsm key
# SELECTION_RATIONALE = ""   # ← 可选：记录选择理由

# 列出所有已成功运行的嵌入方法，供研究者参考
print("\n===== 已成功运行的嵌入 =====")
_available = [k for k in ["X_pca", "X_pca_harmony", "X_scVI", "X_scANVI", "X_scCRAFT"]
              if k in adata.obsm]
for _k in _available:
    _shape = adata.obsm[_k].shape
    print(f"  {_k}: {_shape[0]} cells x {_shape[1]} dims")
if not _available:
    print("  无可用嵌入——请检查上方方法 cell 的执行状态")
print()

# 验证研究者选择（仅当 SELECTED_EMBEDDING 非 None 时执行）
if SELECTED_EMBEDDING is not None:
    print(f"研究者选择: {SELECTED_EMBEDDING}")
    if SELECTED_EMBEDDING not in adata.obsm:
        raise ValueError(
            f"selected_embedding={SELECTED_EMBEDDING!r} 不在 adata.obsm 中。\n"
            f"  可用嵌入: {list(adata.obsm.keys())}\n"
            "  请确认嵌入方法已成功运行，或改回 SELECTED_EMBEDDING = None。"
        )
    # 写入决策到 adata.uns——下游 05 读取此键确定使用的嵌入空间
    adata.uns["selected_embedding"] = SELECTED_EMBEDDING
    adata.uns["selection_rationale"] = SELECTION_RATIONALE
    _reason_display = SELECTION_RATIONALE if SELECTION_RATIONALE else "(未填写)"
    print(f"  选择理由: {_reason_display}")
    print("  已写入 adata.uns[\"selected_embedding\"] —— 05 聚类将使用此嵌入空间。")
else:
    print("⚠️ 尚未选择 selected_embedding。")
    print("  修改本 cell 上方的 SELECTED_EMBEDDING = None 为实际嵌入 key（如 X_scVI）")
    print("  然后重新运行本 cell。")
    print("  Stage 状态将保持 NEEDS_REVIEW，不会自动 promote。")

_decision_completed = True
print("\n决策 cell 完成。")

## 运行元数据——plain `adata.uns` 写入

版本化键（`harmony_v1`、`scvi_v1` 等）记录每个方法以什么参数运行。
PI 用于追踪溯源和重新运行。

In [ ]:
# 记录每个方法的运行元数据（版本化键，支持重跑时共存）。
print("\n===== 运行元数据 =====")

if "X_pca" in adata.obsm:
    adata.uns["pca_v1"] = {
        "method": "pca",
        "n_comps_computed": N_PCS,      # 总共计算了多少 PC（用于 elbow 诊断）
        "n_pcs_used": N_PCS_USE,        # 实际送入下游的维度数
        "use_hvg": True,
        "svd_solver": "arpack",
        "obsm_key": "X_pca",
        "timestamp": datetime.datetime.now().isoformat(),
    }

if "X_pca_harmony" in adata.obsm:
    adata.uns["harmony_v1"] = {
        "method": "harmony",
        "batch_key": BATCH_KEY,
        "n_pcs_input": N_PCS_USE,       # 喂给 Harmony 的 PCA 维度数
        "n_pcs_output": adata.obsm["X_pca_harmony"].shape[1],  # Harmony 输出维度数
        "obsm_key": "X_pca_harmony",
        "converged": _converged,
        "timestamp": datetime.datetime.now().isoformat(),
    }

if "X_scVI" in adata.obsm:
    adata.uns["scvi_v1"] = {
        "method": "scVI",
        "batch_key": BATCH_KEY,
        "n_latent": SCVI_N_LATENT,
        "n_layers": SCVI_N_LAYERS,
        "max_epochs": SCVI_MAX_EPOCHS,
        "gene_likelihood": SCVI_GENE_LIKELIHOOD,
        "early_stopping": SCVI_EARLY_STOPPING,
        "counts_source": _counts_source,  # 实际使用的 counts 来源
        "obsm_key": "X_scVI",
        "timestamp": datetime.datetime.now().isoformat(),
    }

if "X_scANVI" in adata.obsm:
    adata.uns["scanvi_v1"] = {
        "method": "scANVI",
        "batch_key": BATCH_KEY,
        "n_latent": SCVI_N_LATENT,
        "n_epochs": SCANVI_N_EPOCHS,
        "label_key": SCANVI_LABEL_KEY,
        "counts_source": _counts_source,  # 实际使用的 counts 来源
        "obsm_key": "X_scANVI",
        "timestamp": datetime.datetime.now().isoformat(),
    }

if "X_scCRAFT" in adata.obsm:
    adata.uns["sccraft_v1"] = {
        "method": "scCRAFT",
        "batch_key": BATCH_KEY,
        "epochs": SCCRAFT_EPOCHS,
        "d_coef": SCCRAFT_D_COEF,
        "kl_coef": SCCRAFT_KL_COEF,
        "n_top_genes": SCCRAFT_N_TOP_GENES,
        "cluster_method": SCCRAFT_CLUSTER_METHOD,
        "counts_source": _counts_source,
        "obsm_key": "X_scCRAFT",
        "timestamp": datetime.datetime.now().isoformat(),
    }

# 写入有效运行参数供下游审计
adata.uns["04_effective_params"] = snapshot_effective_parameters(
    globals(), exclude=("UPSTREAM_CHECKPOINT", "OUTPUT_PATH"), path_root=Path(_root)
)

# per-method 状态记录（决策 9）
adata.uns["04_method_status"] = {k: v.value for k, v in _method_status.items()}

# 统一追踪字段——stage + version（与上游字段合并，保持 pipeline 编号命名一致）
adata.uns["stage"] = "04_embedded"     # 本 stage 标识
adata.uns["version"] = f"v{OUTPUT_VERSION}"

# 上游溯源信息
adata.uns["upstream"] = [str(UPSTREAM_CHECKPOINT)]
adata.uns["upstream_inputs"] = [upstream_input]
adata.uns["status"] = "NEEDS_REVIEW"  # 本 notebook 不执行科研选择或 promotion

# 全局 counts 来源记录——供后续 stage 审计
adata.uns["counts_source"] = _counts_source if _counts_source else "unknown"

# 已运行的方法汇总。
# 更新 expression_contract——追加 stage 04 的嵌入状态（不修改上游字段）
if "expression_contract" in adata.uns:
    adata.uns["expression_contract"]["embedding_stage"] = "04_embedded"
    adata.uns["expression_contract"]["selected_embedding"] = SELECTED_EMBEDDING
    adata.uns["expression_contract"]["available_embeddings"] = [
        k for k in adata.obsm.keys()
        if k.startswith("X_pca") or k.startswith("X_sc")
    ]
    print("✓ expression_contract 已更新嵌入状态信息")

active_embeddings = [k for k in adata.obsm.keys()
                     if k.startswith("X_pca") or k.startswith("X_sc")]
print(f"已产出的嵌入: {active_embeddings}")
print(f"status: {adata.uns['status']}")
print(f"method_status: {adata.uns['04_method_status']}")
for k in sorted(adata.uns.keys()):
    if k.endswith("_v1"):
        print(f"  {k}: {list(adata.uns[k].keys())}")

## Float32 转换

scVI/scANVI 输出默认 float64。转为 float32 内存减半，
对单细胞数据的有效精度无实质影响。

**为什么 float32 够用？** 单细胞 counts 和嵌入携带的信息精度
远低于 float64 的 15 位有效数字。float64 只是白白浪费内存。

In [ ]:
# 将所有 obsm 潜变量矩阵转为 float32。
print("\n===== Float32 转换 =====")
for key in list(adata.obsm.keys()):
    if adata.obsm[key].dtype != np.float32:
        adata.obsm[key] = adata.obsm[key].astype(np.float32)
        print(f"  obsm['{key}'] 转为 float32")
print("全部 obsm dtype:")
for key in adata.obsm:
    print(f"  obsm['{key}']: shape={adata.obsm[key].shape}, dtype={adata.obsm[key].dtype}")

In [ ]:
# 内存自检——写入前一次断言。
# 守卫最高影响的内存退化：adata.X 变 dense 或丢失 float32。
import scipy.sparse as sp
import numpy as np
assert sp.issparse(adata.X) and adata.X.dtype == np.float32, (
    f"adata.X 不变量违反: sparse={sp.issparse(adata.X)}, dtype={adata.X.dtype}"
)
print("内存自检通过: X 是 sparse CSR float32。")

In [ ]:
# === Stage 04 draft checkpoint：等待研究者选择 ===
# 本轮只保存候选嵌入。没有明确科研选择时固定 NEEDS_REVIEW，绝不 promote。
_embedding_keys = sorted(key for key in adata.obsm if key.startswith("X_"))
_embedding_shapes = {}
_rows_match = True
_dimensions_valid = True
_embeddings_finite = True
for key in _embedding_keys:
    matrix = adata.obsm[key]
    shape = getattr(matrix, "shape", ())
    _embedding_shapes[key] = list(shape)
    _rows_match = _rows_match and len(shape) == 2 and shape[0] == adata.n_obs
    _dimensions_valid = _dimensions_valid and len(shape) == 2 and shape[1] > 0
    values = matrix.data if sp.issparse(matrix) else np.asarray(matrix)
    try:
        _embeddings_finite = _embeddings_finite and bool(np.isfinite(values).all())
    except TypeError:
        _embeddings_finite = False

hard_postconditions = {
    "non_empty": adata.n_obs > 0 and adata.n_vars > 0,
    "output_filename_valid": (
        isinstance(OUTPUT_FILENAME, str)
        and bool(OUTPUT_FILENAME.strip())
        and OUTPUT_FILENAME not in {".", "..", "manifest.json"}
        and not Path(OUTPUT_FILENAME).is_absolute()
        and Path(OUTPUT_FILENAME).name == OUTPUT_FILENAME
        and "/" not in OUTPUT_FILENAME and "\\" not in OUTPUT_FILENAME
        and re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9._-]*\.h5ad", OUTPUT_FILENAME) is not None
        and not any(ord(char) < 32 or ord(char) == 127 for char in OUTPUT_FILENAME)
    ),
    "embedding_present": bool(_embedding_keys),
    "embedding_rows_match": _rows_match,
    "embedding_dimensions_valid": _dimensions_valid,
    "embeddings_finite": _embeddings_finite,
}
_method_required = aggregate_method_status(_method_status)
# 检查是否有部分勾选方法成功、部分失败（SUCCESS_WITH_WARNINGS 条件）
_selected_methods = [k for k, v in _method_status.items() if v != MethodStatus.SKIPPED_BY_USER]
_failed_selected = [k for k, v in _method_status.items()
                     if v == MethodStatus.FAILED and v != MethodStatus.SKIPPED_BY_USER]
_succeeded_selected = [k for k, v in _method_status.items() if v == MethodStatus.SUCCESS]
_has_warnings = len(_failed_selected) > 0 and len(_succeeded_selected) > 0
_needs_review = SELECTED_EMBEDDING is None

# 先校验硬性后置条件（embedding 形状/有限值等），FAILED 立即阻断
stage_status = determine_stage_status(
    _method_required, hard_postconditions, needs_review=False, allow_no_required_methods=True
)
if stage_status == StageStatus.FAILED:
    pass  # 硬性条件不满足，下面 raise 阻断
elif _has_warnings:
    stage_status = StageStatus.SUCCESS_WITH_WARNINGS
elif _needs_review:
    stage_status = StageStatus.NEEDS_REVIEW
effective_parameters = snapshot_effective_parameters(
    globals(), exclude=("UPSTREAM_CHECKPOINT", "OUTPUT_PATH"), path_root=Path(_root)
)
runtime_provenance = collect_runtime_provenance(
    _root, (
        "anndata", "scanpy", "numpy", "pandas", "scipy", "harmonypy",
        "scvi-tools", "torch", "scikit-learn", "sccraft",
    )
)
manifest_payload = {
    "run_id": RUN_ID, "stage": "04_embedded", "stage_status": stage_status.value,
    "inputs": [upstream_input], "effective_parameters": effective_parameters,
    "runtime_provenance": runtime_provenance,
    "hard_postconditions": hard_postconditions, "embedding_shapes": _embedding_shapes,
    "method_status": {k: v.value for k, v in _method_status.items()},
    "selected_embedding": SELECTED_EMBEDDING,
    "selection_rationale": SELECTION_RATIONALE,
    "counts_validation": {
        "valid": _counts_valid,
        "failure_reasons": _counts_failure_reasons,
        "checks": adata.uns.get("scvi_validation", {}).get("checks", {}),
    },
}
run_paths = prepare_run(RUN_ROOT, RUN_ID)
if stage_status.value == "FAILED":
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise RuntimeError(f"Stage 04 FAILED: {hard_postconditions}")

adata.uns["stage"] = "04_embedded"
adata.uns["status"] = stage_status.value
adata.uns["upstream"] = [str(UPSTREAM_CHECKPOINT)]
adata.uns["upstream_inputs"] = [upstream_input]
adata.uns["version"] = f"v{OUTPUT_VERSION}"
adata.uns["run_id"] = RUN_ID
draft_checkpoint = run_paths.draft_dir / OUTPUT_FILENAME
try:
    adata.write_h5ad(draft_checkpoint, compression="lzf")
    checkpoint_sha256 = sha256_file(draft_checkpoint)
except Exception as error:
    draft_checkpoint.unlink(missing_ok=True)
    manifest_payload["stage_status"] = "FAILED"
    manifest_payload["failure"] = {"type": type(error).__name__, "message": str(error)}
    atomic_write_json(run_paths.manifest_path, manifest_payload)
    raise
manifest_payload["checkpoint"] = {"path": OUTPUT_FILENAME, "sha256": checkpoint_sha256}
atomic_write_json(run_paths.manifest_path, manifest_payload)
OUTPUT_PATH = str(validate_checkpoint(run_paths.manifest_path))
print(f"已保存待审查 draft: {OUTPUT_PATH}")
if stage_status.value == "NEEDS_REVIEW":
    print("stage_status: NEEDS_REVIEW；请在上方决策 cell 中设置 SELECTED_EMBEDDING。")
elif stage_status.value == "SUCCESS_WITH_WARNINGS":
    print("stage_status: SUCCESS_WITH_WARNINGS；部分方法失败但至少一种嵌入可用。")
    print(f"  失败方法: {_failed_selected}")
    print(f"  成功方法: {_succeeded_selected}")
elif stage_status.value == "SUCCESS":
    print(f"stage_status: SUCCESS；selected_embedding={SELECTED_EMBEDDING}")

### Stage 04 Verdict

本 stage 遵循 UX-1 模型：研究者逐方法勾选、分别执行、比较结果后在独立决策 cell
设置 `SELECTED_EMBEDDING`。未选 → `NEEDS_REVIEW`；已选 + 部分失败 →
`SUCCESS_WITH_WARNINGS`；已选 + 全部成功 → `SUCCESS`。
审查时应确认：
- [ ] 至少一种嵌入方法成功运行（勾选的方法全部失败为 FAILED）
- [ ] 比较所有已运行嵌入的 UMAP 图（样本混合、细胞类型分离、Marker 叠印）
- [ ] 整合指标表作为定量佐证，不与 UMAP 目测抵触
- [ ] Marker UMAP 各 compartment 清晰分离
- [ ] 无严重应激 PC 主导
- [ ] 研究者在决策 cell 显式选择了 `selected_embedding` 并留有理由


In [ ]:
# 跨 stage 边界释放内存。
# 不释放的话 Jupyter kernel 会一直持有上一 stage 的 AnnData，
# 后续 stage 在同一 kernel 中累积导致 OOM。
del adata
import gc
gc.collect()
print("内存已释放。")

## 🗂 UX-3 Run 管理与跨参数比较

> 此区块只读：枚举当前 `RUN_ROOT` 下所有 run 的状态、参数差异和清理候选。
> **绝不自动删除或提升**——动作（pin/删除）须手动在文件系统执行。
> `pinned.marker` 文件存在于 run 目录下 → 该 run 被保留（PINNED）。


In [ ]:
# === UX-3 wave2 | run 状态总览 ===
# enumerate_run_manifests 只读 json + os.stat，不载入 h5ad 矩阵（内存安全）
from pathlib import Path
import pandas as pd
from scrna_integration.run_contract import (
    enumerate_run_manifests,
    diff_effective_parameters,
    enumerate_cleanup_candidates,
)

try:
    from IPython.display import display as _display
except ImportError:
    _display = print  # 非 Jupyter 环境退化为 print

_run_root = Path(RUN_ROOT)
if not _run_root.exists():
    print(f"⚠️ RUN_ROOT '{_run_root}' 尚不存在，暂无 run 记录。")
    _records = []
else:
    _records = enumerate_run_manifests(_run_root)

if _records:
    _rows = [
        {
            "run_id": r.run_id,
            "state": r.state,
            "category": r.category.name if r.category is not None else "—",
            "pinned": "📌" if r.pinned else "",
            "stage_status": r.stage_status,
            "output_size_mb": (
                f"{r.output_size_bytes / 1e6:.1f}" if r.output_size_bytes is not None else "—"
            ),
            "error": r.error or "",
        }
        for r in _records
    ]
    _display(pd.DataFrame(_rows).to_string(index=False))
    print(f"\n共 {len(_records)} 个 run，其中 pinned: {sum(r.pinned for r in _records)} 个")
else:
    print("ℹ️ 未找到任何 run 记录。")


In [ ]:
# === UX-3 wave2 | 跨 run 参数比较 ===
# diff_effective_parameters 仅展示有差异的参数键，帮助判断哪些 run 用了不同超参
_params_dict = {
    r.run_id: r.effective_parameters
    for r in _records
    if r.effective_parameters is not None
}

if len(_params_dict) < 2:
    print("ℹ️ 有效参数记录少于 2 个，无法比较（需要先成功跑多个 run）。")
else:
    _diff = diff_effective_parameters(_params_dict)
    _differing = _diff.get("differing_keys", [])
    if not _differing:
        print("✅ 所有 run 的有效参数完全一致。")
    else:
        print(f"⚠️ 存在差异的参数（{len(_differing)} 个）：")
        _diff_rows = []
        for _key in _differing:
            _row = {"参数": _key}
            for _rid, _info in _diff["parameters"][_key]["values"].items():
                _row[_rid] = _info["value"] if _info["present"] else "（未设置）"
            _diff_rows.append(_row)
        _display(pd.DataFrame(_diff_rows).to_string(index=False))
    print(f"\n共同参数（全 run 一致）：{len(_diff.get('shared_keys', []))} 个")


In [ ]:
# === UX-3 wave2 | 清理候选枚举 ===
# 只列出 SUPERSEDED/FAILED run 中的大文件（默认 *.h5ad）
# ⚠️ 此处仅枚举，不执行任何删除——请人工确认后手动清理
_candidates = enumerate_cleanup_candidates(_records)

if not _candidates:
    print("✅ 无清理候选（SUPERSEDED/FAILED run 中无大型 .h5ad 文件）。")
else:
    _clean_rows = [
        {
            "run_id": c.run_id,
            "category": c.category.name if c.category is not None else "—",
            "path": str(c.path.name),  # 只显示文件名，路径太长
            "size_mb": f"{c.size_bytes / 1e6:.1f}",
            "state": c.state,
        }
        for c in _candidates
    ]
    _display(pd.DataFrame(_clean_rows).to_string(index=False))
    _total_mb = sum(c.size_bytes for c in _candidates) / 1e6
    print(f"\n⚠️ 共 {len(_candidates)} 个清理候选，总大小约 {_total_mb:.1f} MB")
    print("删除命令示例（人工确认后执行）：")
    for c in _candidates[:3]:  # 只打印前 3 条
        print(f"  rm '{c.path}'")
